# Stable Diffusion 2.1: generazione, filtro e valutazione a 100 inference step

Questo notebook descrive lo studio degli effetti di una generazione con **100 inference step** sulle mammografie sintetiche positive e negative.

La pipeline comprende l'intero flusso sperimentale:

1. individua la root del progetto e centralizza configurazione, percorsi e parametri;
2. verifica e prepara i dataset reali e aumentati senza modificarne permanentemente la struttura;
3. prepara il modello base Stable Diffusion 2.1 e una revisione fissata di Diffusers;
4. riprende il fine-tuning dagli eventuali checkpoint disponibili;
5. valuta tutti i checkpoint sul validation set e seleziona quello con FID medio migliore;
6. genera 2722 immagini RAW per ciascuna classe usando 100 inference step;
7. applica un filtro adattivo non supervisionato, calibrato separatamente sui reali del train;
8. confronta immagini RAW e filtrate sul validation set;
9. calcola FID, Inception Score e PRDC finali sul test set usando le 1361 immagini filtrate per classe;
10. visualizza la diagnostica dei checkpoint e la curva di loss di training;
11. confronta i risultati a 50 e 100 inference step su grandezze omogenee;
12. mostra griglie di campioni positivi e negativi per ispezione qualitativa.

Le operazioni costose sono progettate per essere idempotenti: dataset, checkpoint,
immagini e metriche complete vengono riutilizzati quando possibile. Modello, checkpoint
e immagini generate restano sotto `experiments/`; metriche, grafici e log EcoTracker
del notebook 3b vengono raccolti in `results/03b_finetuning_filtered/`.

Il notebook è idempotente: tutte le operazioni costose (fine-tuning, generazione, valutazione, filtro) verificano la presenza dei risultati e li riutilizzano quando completi. Una riesecuzione completa non rigenera immagini né riaddestra il modello, ma ricalcola soltanto le metriche e i grafici. Non sono necessari flag o configurazioni manuali per rieseguirlo.


## 1. Ambiente e configurazione

Le celle seguenti inizializzano l'ambiente di esecuzione e definiscono una singola
configurazione condivisa dall'intero notebook.

In particolare:

- vengono importate le dipendenze Python necessarie;
- `find_project_root` individua la repository anche quando il notebook viene eseguito
  da percorsi o mount differenti;
- vengono mostrati interprete Python, versione PyTorch e disponibilità della GPU;
- tutti i percorsi sono derivati da `EXPERIMENT_DIR` e non dipendono da path assoluti
  della macchina originale;
- vengono dichiarati prompt, parametri di training, 100 inference step, numerosità
  delle immagini RAW e filtrate, percorsi delle metriche e log di sostenibilità;
- vengono create soltanto le directory necessarie, senza cancellare artefatti esistenti.

Questa sezione costituisce la fonte unica di verità per i parametri utilizzati nelle
celle successive.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from datetime import datetime
import gc
import hashlib
import importlib.util
import json
import os
import re
import shutil
import subprocess
import sys
import zipfile

import matplotlib.pyplot as plt
import pandas as pd
import torch
from PIL import Image

# Verifica se il modulo specifico di Google Colab è disponibile.
IS_COLAB = importlib.util.find_spec("google.colab") is not None

if IS_COLAB:
    from google.colab import drive

    # Monta Google Drive in /content/drive.
    # Se Drive è già montato, Colab non lo rimonta.
    drive.mount("/content/drive", force_remount=False)

    # La cartella MammoDiffusion è condivisa con l'utente.
    # Per renderla accessibile da Colab bisogna aggiungerne
    # una scorciatoia all'interno di "Il mio Drive".
    PROJECT_ROOT_OVERRIDE = "/content/drive/MyDrive/MammoDiffusion"

else:
    print("Ambiente locale o kernel remoto rilevato.")

    # In locale si utilizza la ricerca automatica della root.
    PROJECT_ROOT_OVERRIDE = None

try:
    import gdown
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"])
    import gdown


PROJECT_NAME = "MammoDiffusion"


def find_project_root(project_name=PROJECT_NAME, override=PROJECT_ROOT_OVERRIDE):
    if override is not None:
        root = Path(override).expanduser().resolve()
        if not root.is_dir():
            raise FileNotFoundError(f"PROJECT_ROOT_OVERRIDE non esiste: {root}")
        return root

    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if candidate.name == project_name or (
            (candidate / "data").is_dir() and (candidate / "notebooks").is_dir()
        ):
            return candidate

    fallback_candidates = [
        cwd / project_name,
        Path("/content") / project_name,
        Path("/content/drive/MyDrive") / project_name,
        Path.home() / project_name,
        Path.home() / "Progetto" / project_name,
    ]
    for candidate in fallback_candidates:
        if candidate.is_dir():
            return candidate.resolve()

    raise FileNotFoundError(
        "Root di MammoDiffusion non trovata. Esegui il notebook dalla repository "
        "oppure imposta PROJECT_ROOT_OVERRIDE."
    )


PROJECT_ROOT = find_project_root()
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA disponibile:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "nessuna")
print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
# Dataset condivisi dal progetto
DATA_DIR = PROJECT_ROOT / "data"
ARCHIVES_DIR = DATA_DIR / "archives"
DATA_PROCESSED_DIR = DATA_DIR / "processed"
DATA_AUG = DATA_DIR / "real_augmented"

PROCESSED_DRIVE_ID = "1qQral_BIBlMl0QN3PllJukdYTOmNGWr3"
AUGMENTED_DRIVE_ID = "1XRc0SxLEPP-zbMJApn4ruaiH8u_rDc-0"
PROCESSED_ZIP_PATH = ARCHIVES_DIR / "processed.zip"
AUGMENTED_ZIP_PATH = ARCHIVES_DIR / "real_augmented.zip"

# Esperimento e modello
EXPERIMENT_NAME = "20260611_sd21_rsna_mlo_512_inference_100_steps"
EXPERIMENTS_DIR = PROJECT_ROOT / "experiments"
EXPERIMENT_DIR = EXPERIMENTS_DIR / EXPERIMENT_NAME

SD21_MODEL_DRIVE_ID = "10XRn-bxpp7tP6ROWLYpeCNYJZIaHfMUt"
PRETRAINED_MODEL_DIR = EXPERIMENT_DIR / "pretrained_model" / "stable-diffusion-2-1-base"
PRETRAINED_MODEL_ZIP_PATH = EXPERIMENT_DIR / "archives" / "stable-diffusion-2-1-base.zip"
FORCE_MODEL_REDOWNLOAD = False

DIFFUSERS_REPO_DIR = EXPERIMENT_DIR / "diffusers_repo"
DIFFUSERS_REVISION = "3759fab56d3170a04d747e918a13e55fda6681e2"
HF_CACHE_DIR = EXPERIMENT_DIR / "hf_cache"
SD_OUTPUT_DIR = EXPERIMENT_DIR / "model"

# Prompt condizionati dalle label
POSITIVE_PROMPT = (
    "grayscale MLO mammogram, breast cancer positive, malignant finding, "
    "suspicious lesion, medical imaging"
)
NEGATIVE_PROMPT = (
    "grayscale MLO mammogram, breast cancer negative, no malignant finding, "
    "normal screening mammogram, medical imaging"
)

# Fine-tuning
RESOLUTION = 512
TRAIN_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 1e-5
MAX_TRAIN_STEPS = 8000
CHECKPOINTING_STEPS = 500
CHECKPOINTS_TOTAL_LIMIT = 32
RESUME_FROM_CHECKPOINT = "latest"
TRAIN_SEED = 42

# Valutazione e generazione
N_EVAL_IMAGES_PER_CLASS = 100
N_VALIDATION_IMAGES_PER_CLASS = 73
N_TEST_IMAGES_PER_CLASS = 73
INFERENCE_STEPS = 100
EVAL_GUIDANCE_SCALE = 7.5
EVAL_SEED = 42
PRDC_NEAREST_K = 5
N_FINAL_IMAGES_PER_CLASS = 2722
FINAL_GENERATE_CLASSES = ["positive", "negative"]
N_SELECTED_PER_CLASS = 1361
NONBLACK_THRESHOLD = 10

VALIDATION_METADATA_PATH = DATA_PROCESSED_DIR / "metadata" / "val.csv"
TEST_METADATA_PATH = DATA_PROCESSED_DIR / "metadata" / "test.csv"
TRAIN_METADATA_PATH = DATA_PROCESSED_DIR / "metadata" / "train.csv"

RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_03B_DIR = RESULTS_DIR / "03b_finetuning_filtered"
METRICS_DIR = RESULTS_03B_DIR / "metrics"
PLOTS_DIR = RESULTS_03B_DIR / "plots"
ECOTRACKER_DIR = RESULTS_03B_DIR / "ecotracker"

EVAL_DIR = EXPERIMENT_DIR / "eval_checkpoints"
EVAL_METRICS_PATH = METRICS_DIR / "checkpoint_validation_metrics.json"
EVAL_SUSTAINABILITY_LOG = ECOTRACKER_DIR / "sustainability_validation.jsonl"

FINAL_GEN_DIR = EXPERIMENT_DIR / "generated_images" / "final"
FINAL_NEG_DIR = FINAL_GEN_DIR / "negative"
FINAL_POS_DIR = FINAL_GEN_DIR / "positive"
FINAL_DIRS = {"negative": FINAL_NEG_DIR, "positive": FINAL_POS_DIR}
FINAL_CLASS_LABELS = {"negative": 0, "positive": 1}

FILTERED_DIRS = {
    class_name: FINAL_GEN_DIR / f"{class_name}_filtered_{N_SELECTED_PER_CLASS}_adaptive_mask"
    for class_name in FINAL_GENERATE_CLASSES
}
FILTER_REPORT_PATHS = {
    class_name: METRICS_DIR / f"filter_report_{class_name}_adaptive_mask.csv"
    for class_name in FINAL_GENERATE_CLASSES
}
FILTER_SUMMARY_PATHS = {
    class_name: METRICS_DIR / f"filter_summary_{class_name}_adaptive_mask.json"
    for class_name in FINAL_GENERATE_CLASSES
}

VALIDATION_COMPARISON_CSV = METRICS_DIR / "validation_comparison_raw_vs_filtered.csv"
VALIDATION_COMPARISON_JSON = METRICS_DIR / "validation_comparison_raw_vs_filtered.json"
FINAL_TEST_METRICS_PATH = METRICS_DIR / "final_test_metrics.json"
FINAL_TEST_METRICS_CSV = METRICS_DIR / "final_test_metrics.csv"
FINAL_TEST_RAW_METRICS_PATH = METRICS_DIR / "final_test_metrics_raw_2722.json"
LEGACY_FINAL_TEST_RAW_METRICS_PATH = EXPERIMENT_DIR / "final_test_metrics_raw_2722.json"

SUSTAINABILITY_LOG = ECOTRACKER_DIR / "sustainability_finetuning.jsonl"
FINAL_GENERATION_LOG = ECOTRACKER_DIR / "sustainability_generation.jsonl"
GENERATION_INFO_PATH = METRICS_DIR / "generation_info.json"

for directory in [
    ARCHIVES_DIR,
    DATA_AUG,
    EXPERIMENT_DIR,
    PRETRAINED_MODEL_ZIP_PATH.parent,
    HF_CACHE_DIR,
    SD_OUTPUT_DIR,
    EVAL_DIR,
    METRICS_DIR,
    PLOTS_DIR,
    ECOTRACKER_DIR,
    FINAL_NEG_DIR,
    FINAL_POS_DIR,
    *FILTERED_DIRS.values(),
]:
    directory.mkdir(parents=True, exist_ok=True)


def label_to_prompt(label):
    prompts = {0: NEGATIVE_PROMPT, 1: POSITIVE_PROMPT}
    try:
        return prompts[int(label)]
    except KeyError as exc:
        raise ValueError(f"Label non valida: {label}") from exc


print("Esperimento:", EXPERIMENT_NAME)
print("Cartella esperimento:", EXPERIMENT_DIR)
print("Cartella risultati 3b:", RESULTS_03B_DIR)
print("Inference step:", INFERENCE_STEPS)

## 2. Preparazione e verifica dei dati

Le celle successive verificano la disponibilità dei dataset persistenti in `data/processed/` e `data/real_augmented/`. Se un dataset manca o risulta incompleto, viene scaricato dal Drive ed estratto nella posizione prevista.

La preparazione fa in modo che le immagini reali e aumentate restano nelle rispettive directory originali e vengono risolte in modo **source-aware**:

- i record con `source="real"` usano `data/processed/train/<label>/`;
- i record aumentati usano `data/real_augmented/`;

Vengono inoltre controllate le colonne del metadata, normalizzate label e caption e mostrati distribuzione delle classi, sorgenti e un campione visivo. La funzione `stage_training_dataset` prepara successivamente una copia temporanea autosufficiente per Diffusers, senza mutare i dataset persistenti.


In [ ]:
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
EXPECTED_SPLITS = ("train", "val", "test")
EXPECTED_LABELS = ("0", "1")


def count_images(directory):
    directory = Path(directory)
    return sum(
        path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
        for path in directory.rglob("*")
    ) if directory.is_dir() else 0


def download_zip(drive_id, destination, force=False):
    destination = Path(destination)
    if destination.exists() and not force and zipfile.is_zipfile(destination):
        print("Archivio già presente:", destination)
        return
    destination.unlink(missing_ok=True)
    gdown.download(id=drive_id, output=str(destination), quiet=False)
    if not destination.exists() or not zipfile.is_zipfile(destination):
        raise RuntimeError(f"Download non valido: {destination}")


def processed_dataset_ready(directory):
    directory = Path(directory)
    return all(
        count_images(directory / split / label) > 0
        for split in EXPECTED_SPLITS
        for label in EXPECTED_LABELS
    )


def find_directory(root, predicate, description):
    root = Path(root)
    for candidate in [root, *(path for path in root.rglob("*") if path.is_dir())]:
        if predicate(candidate):
            return candidate
    raise FileNotFoundError(f"Directory {description} non trovata dentro {root}")


def prepare_processed_dataset():
    if processed_dataset_ready(DATA_PROCESSED_DIR):
        print("Dataset processed già pronto.")
        return

    download_zip(PROCESSED_DRIVE_ID, PROCESSED_ZIP_PATH)
    with TemporaryDirectory(prefix="mammo_processed_extract_") as tmp:
        with zipfile.ZipFile(PROCESSED_ZIP_PATH) as archive:
            archive.extractall(tmp)
        source_dir = find_directory(tmp, processed_dataset_ready, "processed")
        shutil.copytree(source_dir, DATA_PROCESSED_DIR, dirs_exist_ok=True)

    if not processed_dataset_ready(DATA_PROCESSED_DIR):
        raise FileNotFoundError("Dataset processed incompleto dopo l'estrazione.")


def augmented_dataset_ready():
    return (DATA_AUG / "metadata.csv").is_file() and count_images(DATA_AUG) > 0


def prepare_augmented_dataset():
    if augmented_dataset_ready():
        print("Dataset augmented già pronto.")
        return

    download_zip(AUGMENTED_DRIVE_ID, AUGMENTED_ZIP_PATH)
    with TemporaryDirectory(prefix="mammo_augmented_extract_") as tmp:
        with zipfile.ZipFile(AUGMENTED_ZIP_PATH) as archive:
            archive.extractall(tmp)
        source_dir = find_directory(
            tmp,
            lambda path: (path / "metadata.csv").is_file() and count_images(path) > 0,
            "augmented con metadata.csv",
        )
        DATA_AUG.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source_dir / "metadata.csv", DATA_AUG / "metadata.csv")
        for image_path in source_dir.rglob("*"):
            if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:
                destination = DATA_AUG / image_path.name
                if not destination.exists():
                    shutil.copy2(image_path, destination)

    if not augmented_dataset_ready():
        raise FileNotFoundError("Dataset augmented incompleto dopo l'estrazione.")


def load_training_metadata(data_aug):
    metadata_path = Path(data_aug) / "metadata.csv"
    metadata = pd.read_csv(metadata_path).copy()
    required = {"file_name", "label"}
    missing = required.difference(metadata.columns)
    if missing:
        raise ValueError(f"Colonne mancanti in {metadata_path}: {sorted(missing)}")
    metadata["file_name"] = metadata["file_name"].astype(str).str.replace("\\", "/", regex=False)
    metadata["label"] = metadata["label"].astype(int)
    metadata["text"] = metadata["label"].map(label_to_prompt)
    return metadata


def is_valid_training_image(path):
    path = Path(path)
    return (
        path.is_file()
        and not path.is_symlink()
        and path.suffix.lower() in IMAGE_EXTENSIONS
    )


def resolve_training_image_path(row):
    file_name = Path(str(row["file_name"]))
    label = str(int(row["label"]))
    source = str(row.get("source", "")).strip().lower()
    real_candidate = DATA_PROCESSED_DIR / "train" / label / file_name.name
    augmented_candidate = DATA_AUG / file_name.name

    if source == "real":
        candidates = [real_candidate]
    elif source in {"positive_augmentation", "augmentation", "augmented"}:
        candidates = [augmented_candidate]
    else:
        candidates = [augmented_candidate, real_candidate, PROJECT_ROOT / file_name]

    original_value = row.get("original_processed_path")
    if source == "real" and pd.notna(original_value) and str(original_value).strip():
        original = Path(str(original_value))
        if "data" in original.parts:
            candidates.append(PROJECT_ROOT / Path(*original.parts[original.parts.index("data"):]))
        candidates.append(original)

    for candidate in candidates:
        if is_valid_training_image(candidate):
            return candidate.resolve()

    checked = ", ".join(str(path) for path in candidates)
    raise FileNotFoundError(
        f"Immagine valida non trovata per file_name={row['file_name']}, source={source}. "
        f"Percorsi controllati: {checked}"
    )


def stage_training_dataset(metadata_df, staging_dir):
    """Copia ogni campione in uno staging temporaneo compatibile con HF imagefolder."""
    staging_dir = Path(staging_dir)
    staged = metadata_df.copy()
    staged_names = []

    for index, (_, row) in enumerate(staged.iterrows()):
        source_path = resolve_training_image_path(row)
        destination_name = f"image_{index:06d}{source_path.suffix.lower()}"
        shutil.copy2(source_path, staging_dir / destination_name)
        staged_names.append(destination_name)

    staged["file_name"] = staged_names
    staged.to_csv(staging_dir / "metadata.csv", index=False)

    if count_images(staging_dir) != len(staged):
        raise RuntimeError("Lo staging temporaneo non contiene tutti i campioni attesi.")
    return staged

In [ ]:
prepare_processed_dataset()
prepare_augmented_dataset()
metadata_df = load_training_metadata(DATA_AUG)

print("Campioni training:", len(metadata_df))
print("\nDistribuzione label:")
print(metadata_df["label"].value_counts().sort_index())
if "source" in metadata_df.columns:
    print("\nDistribuzione source:")
    print(metadata_df["source"].value_counts())

sample_df = metadata_df.sample(min(6, len(metadata_df)), random_state=42)
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for axis, (_, row) in zip(axes.flat, sample_df.iterrows()):
    image_path = resolve_training_image_path(row)
    with Image.open(image_path) as image:
        axis.imshow(image.convert("L"), cmap="gray")
    axis.set_title(f'label={row["label"]} | source={row.get("source", "n/a")}')
    axis.axis("off")
for axis in axes.flat[len(sample_df):]:
    axis.axis("off")
plt.tight_layout()
plt.show()

## 3. Preparazione del modello base e di Diffusers

Le seguenti celle preparano tutte le dipendenze necessarie al fine-tuning e alla generazione, mantenendole associate all'esperimento.

Per il modello Stable Diffusion 2.1 viene verificata la presenza della struttura Diffusers completa (`model_index.json`, scheduler, tokenizer, text encoder, VAE e UNet). Se necessario, il modello viene scaricato ed estratto; gli alias dei pesi vengono creati tramite **copie fisiche**.

Per Diffusers:

- vengono installati soltanto i pacchetti Python mancanti;
- il repository locale viene clonato solo se assente;
- viene usata la revisione fissata `3759fab...` per rendere riproducibile il training;
- l'installazione editable viene resa importabile anche nel kernel corrente;
- una revisione differente produce un avviso, senza bloccare automaticamente il notebook.

Al termine vengono inizializzati il monitoraggio di sostenibilità e l'evaluator usato per FID e Inception Score.


In [ ]:
MODEL_WEIGHT_ALIASES = {
    "text_encoder": ("model.fp16.safetensors", "model.safetensors"),
    "unet": ("diffusion_pytorch_model.fp16.safetensors", "diffusion_pytorch_model.safetensors"),
    "vae": ("diffusion_pytorch_model.fp16.safetensors", "diffusion_pytorch_model.safetensors"),
}


def has_diffusers_structure(model_dir):
    model_dir = Path(model_dir)
    required = ("model_index.json", "scheduler", "tokenizer", "text_encoder", "vae", "unet")
    return all((model_dir / item).exists() for item in required)


def local_sd_model_ready(model_dir):
    model_dir = Path(model_dir)
    weights = (
        model_dir / "text_encoder" / "model.safetensors",
        model_dir / "unet" / "diffusion_pytorch_model.safetensors",
        model_dir / "vae" / "diffusion_pytorch_model.safetensors",
    )
    return has_diffusers_structure(model_dir) and all(path.is_file() for path in weights)


def create_model_weight_copies(model_dir):
    """Crea sempre copie fisiche con i nomi standard attesi da Diffusers."""
    model_dir = Path(model_dir)
    for subfolder, (source_name, target_name) in MODEL_WEIGHT_ALIASES.items():
        source_path = model_dir / subfolder / source_name
        target_path = model_dir / subfolder / target_name
        if target_path.is_symlink():
            target_path.unlink()
        elif target_path.exists():
            continue
        if not source_path.is_file():
            raise FileNotFoundError(f"Peso sorgente non trovato: {source_path}")
        shutil.copy2(source_path, target_path)


def prepare_pretrained_model():
    if local_sd_model_ready(PRETRAINED_MODEL_DIR) and not FORCE_MODEL_REDOWNLOAD:
        print("Modello Stable Diffusion 2.1 già pronto.")
        return PRETRAINED_MODEL_DIR.resolve()

    download_zip(SD21_MODEL_DRIVE_ID, PRETRAINED_MODEL_ZIP_PATH, FORCE_MODEL_REDOWNLOAD)
    with TemporaryDirectory(prefix="mammo_sd21_extract_") as tmp:
        with zipfile.ZipFile(PRETRAINED_MODEL_ZIP_PATH) as archive:
            archive.extractall(tmp)
        source_dir = find_directory(tmp, has_diffusers_structure, "modello Diffusers")
        if PRETRAINED_MODEL_DIR.exists():
            shutil.rmtree(PRETRAINED_MODEL_DIR)
        shutil.copytree(source_dir, PRETRAINED_MODEL_DIR)
        create_model_weight_copies(PRETRAINED_MODEL_DIR)

    if not local_sd_model_ready(PRETRAINED_MODEL_DIR):
        raise FileNotFoundError("Modello Stable Diffusion 2.1 incompleto dopo l'estrazione.")
    return PRETRAINED_MODEL_DIR.resolve()


LOCAL_MODEL_DIR = prepare_pretrained_model()
print("Modello locale:", LOCAL_MODEL_DIR)

In [ ]:
# Prepara il repository Diffusers locale e le dipendenze necessarie al training.
REQUIRED_PACKAGES = {
    "diffusers": "diffusers",
    "transformers": "transformers",
    "accelerate": "accelerate",
    "datasets": "datasets",
    "safetensors": "safetensors",
    "huggingface_hub": "huggingface_hub",
    "bitsandbytes": "bitsandbytes",
    "prdc": "prdc",
    "tensorboard": "tensorboard",
}
missing_packages = [
    package_name
    for module_name, package_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]
if missing_packages:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", *missing_packages
    ])

if not (DIFFUSERS_REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "https://github.com/huggingface/diffusers", str(DIFFUSERS_REPO_DIR)],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(DIFFUSERS_REPO_DIR), "checkout", DIFFUSERS_REVISION],
        check=True,
    )

TRAIN_SCRIPT = DIFFUSERS_REPO_DIR / "examples" / "text_to_image" / "train_text_to_image.py"
if not TRAIN_SCRIPT.is_file():
    raise FileNotFoundError(f"Script di training non trovato: {TRAIN_SCRIPT}")

current_revision = subprocess.check_output(
    [
        "git",
        "-c", f"safe.directory={DIFFUSERS_REPO_DIR}",
        "-C", str(DIFFUSERS_REPO_DIR),
        "rev-parse", "HEAD",
    ],
    text=True,
).strip()
if current_revision != DIFFUSERS_REVISION:
    print(f"ATTENZIONE: Diffusers è a {current_revision}, atteso {DIFFUSERS_REVISION}.")

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-e", str(DIFFUSERS_REPO_DIR)
])

# Rende l'installazione editable importabile anche senza riavviare il kernel.
DIFFUSERS_SRC_DIR = DIFFUSERS_REPO_DIR / "src"
if str(DIFFUSERS_SRC_DIR) not in sys.path:
    sys.path.insert(0, str(DIFFUSERS_SRC_DIR))
importlib.invalidate_caches()

for name in [
    "HF_HUB_VERBOSITY",
    "TRANSFORMERS_VERBOSITY",
    "DIFFUSERS_VERBOSITY",
    "ACCELERATE_LOG_LEVEL",
]:
    os.environ[name] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONWARNINGS"] = "ignore"

from eco_tracker import measure_sustainability
from generative_evaluator_finetuning import GenerativeEvaluator
from prdc import compute_prdc

print("Diffusers revision:", current_revision)
print("Script training:", TRAIN_SCRIPT)

## 4. Fine-tuning

La cella successiva costruisce ed esegue il comando di fine-tuning text-to-image con i parametri centralizzati nella configurazione.

Prima dell'avvio, tutti i 3061 campioni di training vengono copiati in una directory temporanea compatibile con Hugging Face, `imagefolder`, insieme a un nuovo `metadata.csv`. Questo staging isola il training dalla struttura persistente dei dati e viene eliminato tramite `try/finally` anche in caso di errore.

Il training utilizza precisione FP16, gradient accumulation, gradient checkpointing e 8-bit Adam. Con `RESUME_FROM_CHECKPOINT="latest"` riprende automaticamente dall'ultimo checkpoint disponibile; impostando il valore a `None` partirebbe invece da zero.

L'esito del subprocess e le misure di tempo, RAM, energia e CO2 vengono aggiunti a `results/03b_finetuning_filtered/ecotracker/sustainability_finetuning.jsonl`. Se il processo termina con codice diverso da zero, la cella propaga esplicitamente l'errore.


In [ ]:
def build_training_command(train_data_dir):
    command = [
        sys.executable, "-m", "accelerate.commands.launch",
        "--mixed_precision=fp16",
        "--num_processes=1",
        str(TRAIN_SCRIPT),
        "--pretrained_model_name_or_path", str(LOCAL_MODEL_DIR),
        "--train_data_dir", str(train_data_dir),
        "--image_column", "image",
        "--caption_column", "text",
        "--resolution", str(RESOLUTION),
        "--center_crop",
        "--train_batch_size", str(TRAIN_BATCH_SIZE),
        "--gradient_accumulation_steps", str(GRADIENT_ACCUMULATION_STEPS),
        "--gradient_checkpointing",
        "--max_train_steps", str(MAX_TRAIN_STEPS),
        "--learning_rate", str(LEARNING_RATE),
        "--lr_scheduler", "constant",
        "--lr_warmup_steps", "0",
        "--max_grad_norm", "1",
        "--use_8bit_adam",
        "--checkpointing_steps", str(CHECKPOINTING_STEPS),
        "--checkpoints_total_limit", str(CHECKPOINTS_TOTAL_LIMIT),
        "--validation_prompts", POSITIVE_PROMPT, NEGATIVE_PROMPT,
        "--validation_epochs", "2",
        "--seed", str(TRAIN_SEED),
        "--cache_dir", str(HF_CACHE_DIR),
        "--output_dir", str(SD_OUTPUT_DIR),
        "--report_to", "tensorboard",
        "--logging_dir", str(SD_OUTPUT_DIR / "logs"),
        "--dataloader_num_workers", "0",
    ]
    if RESUME_FROM_CHECKPOINT is not None:
        command.extend(["--resume_from_checkpoint", RESUME_FROM_CHECKPOINT])
    return command


def build_training_environment():
    environment = os.environ.copy()
    environment["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    conda_prefix = Path(sys.prefix)
    cuda_lib_paths = [
        conda_prefix / "lib",
        conda_prefix / "targets" / "x86_64-linux" / "lib",
        conda_prefix / "lib" / "python3.11" / "site-packages" / "nvidia" / "cu13" / "lib",
        conda_prefix / "lib" / "python3.11" / "site-packages" / "nvidia" / "nvjitlink" / "lib",
    ]
    existing = environment.get("LD_LIBRARY_PATH", "")
    environment["LD_LIBRARY_PATH"] = ":".join(
        [str(path) for path in cuda_lib_paths if path.exists()] + [existing]
    )
    return environment


run_label = f"finetune_sd21_to_{MAX_TRAIN_STEPS}_steps"
if RESUME_FROM_CHECKPOINT is not None:
    run_label += f"_resume_{RESUME_FROM_CHECKPOINT}"
temporary_training = TemporaryDirectory(prefix="mammo_sd21_train_copy_")
try:
    temporary_train_dir = Path(temporary_training.name)
    staged_metadata = stage_training_dataset(metadata_df, temporary_train_dir)
    command = build_training_command(temporary_train_dir)

    print("Staging temporaneo:", temporary_train_dir)
    print("Campioni copiati:", len(staged_metadata))
    print("Comando fine-tuning:\n", " ".join(map(str, command)))

    with measure_sustainability(label=run_label, sample_interval=0.5) as eco:
        process = subprocess.Popen(
            command,
            env=build_training_environment(),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in iter(process.stdout.readline, ""):
            print(line, end="", flush=True)
        process.wait()
finally:
    temporary_training.cleanup()
    print("Staging temporaneo eliminato.")

record = eco.metrics.to_dict()
record.update({
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "max_train_steps": MAX_TRAIN_STEPS,
    "resume_from_checkpoint": RESUME_FROM_CHECKPOINT,
    "resolution": RESOLUTION,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": LEARNING_RATE,
    "source_metadata": str(DATA_AUG / "metadata.csv"),
    "staging_strategy": "temporary_copy",
    "output_dir": str(SD_OUTPUT_DIR),
    "returncode": process.returncode,
    "status": "completed" if process.returncode == 0 else "failed",
})
with SUSTAINABILITY_LOG.open("a", encoding="utf-8") as handle:
    handle.write(json.dumps(record, ensure_ascii=False) + "\n")

print("Metriche sostenibilità:", eco.metrics)
if process.returncode != 0:
    raise subprocess.CalledProcessError(process.returncode, command)

## 5. Valutazione dei checkpoint e selezione sul validation set

Le celle successive valutano tutti i checkpoint salvati, senza utilizzare il test set
per prendere decisioni sul modello.

La sezione:

- configura pipeline, metadata di train/validation/test e directory degli output;
- costruisce riferimenti reali temporanei tramite copie fisiche, ricostruendo i percorsi
  anche quando nei CSV è rimasta una vecchia root assoluta;
- scopre e ordina i checkpoint `checkpoint-<step>` contenenti un UNet valido;
- genera 100 immagini per classe e checkpoint usando **100 inference step**;
- salta la generazione delle immagini già presenti e libera sempre la VRAM con
  `try/finally`;
- calcola FID, Inception Score e PRDC separatamente per negative e positive, più le medie;
- riusa `results/03b_finetuning_filtered/metrics/checkpoint_validation_metrics.json`
  quando contiene già tutti i checkpoint, purché contenga anche tutte le metriche PRDC,
  evitando ricalcoli non necessari;
- seleziona come `BEST_CHECKPOINT` quello con FID medio più basso sul validation set.

Le metriche PRDC descrivono fedeltà e copertura: precision e density misurano quanto
le immagini generate ricadano vicino alla distribuzione reale, mentre recall e coverage
misurano quanta parte della variabilità reale venga rappresentata. Le stesse feature
Inception estratte per il FID vengono riutilizzate, senza una seconda estrazione. Il
parametro `PRDC_NEAREST_K = 5` è fissato nella configurazione ed è condiviso da tutte
le valutazioni.

Poiché ciascuna classe dispone di 73 riferimenti reali su validation e test, le stime
PRDC possono avere una varianza non trascurabile. Sono particolarmente utili per
confronti relativi condotti sullo stesso riferimento; i valori assoluti devono invece
essere interpretati con cautela.

Le immagini di valutazione sono conservate in `experiments/.../eval_checkpoints/`; il
test set resta riservato esclusivamente alla valutazione finale.


In [ ]:
# Configurazione della selezione checkpoint e della generazione finale
from diffusers import StableDiffusionPipeline, UNet2DConditionModel

for metadata_path in [VALIDATION_METADATA_PATH, TEST_METADATA_PATH, TRAIN_METADATA_PATH]:
    if not metadata_path.is_file():
        raise FileNotFoundError(f"Metadata split non trovato: {metadata_path}")

print("Checkpoint sorgente       :", SD_OUTPUT_DIR)
print("Validation metadata       :", VALIDATION_METADATA_PATH)
print("Test metadata             :", TEST_METADATA_PATH)
print("Immagini generate/ckpt/cls:", N_EVAL_IMAGES_PER_CLASS)
print("Immagini finali/classe    :", N_FINAL_IMAGES_PER_CLASS)

In [ ]:
# Riferimenti reali temporanei per validation/test, sempre tramite copia
from contextlib import contextmanager


def resolve_split_image_path(raw_path, split_name, label):
    original = Path(str(raw_path)).expanduser()
    candidates = [original] if original.is_absolute() else [PROJECT_ROOT / original]
    if "data" in original.parts:
        candidates.append(PROJECT_ROOT / Path(*original.parts[original.parts.index("data"):]))
    candidates.append(DATA_PROCESSED_DIR / split_name / str(int(label)) / original.name)

    for candidate in dict.fromkeys(candidates):
        if candidate.is_file():
            return candidate.resolve()
    checked = "\n".join(f"  - {candidate}" for candidate in dict.fromkeys(candidates))
    raise FileNotFoundError(f"Immagine non trovata per {raw_path}. Percorsi controllati:\n{checked}")


def get_real_image_paths_from_split_metadata(metadata_path, label, n_images, seed=42):
    metadata_path = Path(metadata_path)
    metadata = pd.read_csv(metadata_path)
    required = {"label", "processed_path"}
    missing = required.difference(metadata.columns)
    if missing:
        raise ValueError(f"Colonne mancanti in {metadata_path}: {sorted(missing)}")

    subset = metadata[metadata["label"].astype(int) == int(label)]
    if subset.empty:
        raise ValueError(f"Nessuna immagine label={label} in {metadata_path}")
    n_take = len(subset) if n_images is None else min(int(n_images), len(subset))
    if n_images is not None and n_take < n_images:
        print(f"Richieste {n_images} immagini label={label}; disponibili {n_take}.")

    subset = subset.sample(n=n_take, random_state=seed)
    return [
        resolve_split_image_path(raw_path, metadata_path.stem.lower(), label)
        for raw_path in subset["processed_path"]
    ]


@contextmanager
def temporary_real_reference_dir_from_split_metadata(metadata_path, label, n_images, seed=42):
    image_paths = get_real_image_paths_from_split_metadata(metadata_path, label, n_images, seed)
    with TemporaryDirectory(prefix=f"real_ref_label{label}_") as tmp:
        tmp_dir = Path(tmp)
        for index, source_path in enumerate(image_paths):
            destination = tmp_dir / f"real_{label}_{index:04d}{source_path.suffix.lower()}"
            shutil.copy2(source_path, destination)
        yield tmp_dir


@contextmanager
def temporary_real_reference_dirs_from_split_metadata(metadata_path, n_per_class, seed=42):
    with temporary_real_reference_dir_from_split_metadata(
        metadata_path, label=0, n_images=n_per_class, seed=seed
    ) as real_neg_dir, temporary_real_reference_dir_from_split_metadata(
        metadata_path, label=1, n_images=n_per_class, seed=seed + 1
    ) as real_pos_dir:
        yield real_neg_dir, real_pos_dir


print("Selezione checkpoint: validation.")
print("Valutazione finale: test, esclusivamente nell'ultima cella.")

In [ ]:
# Utility per checkpoint, generazione e metriche
def discover_checkpoints(output_dir):
    checkpoints = []
    for path in Path(output_dir).glob("checkpoint-*"):
        match = re.fullmatch(r"checkpoint-(\d+)", path.name)
        if match and path.is_dir() and (path / "unet").is_dir():
            checkpoints.append((int(match.group(1)), path))
    return sorted(checkpoints)


def load_pipeline_from_checkpoint(checkpoint_path, base_model_dir, device="cuda"):
    unet = UNet2DConditionModel.from_pretrained(
        str(Path(checkpoint_path) / "unet"),
        torch_dtype=torch.float16,
    )
    pipeline = StableDiffusionPipeline.from_pretrained(
        str(base_model_dir),
        unet=unet,
        torch_dtype=torch.float16,
        safety_checker=None,
        requires_safety_checker=False,
    ).to(device)
    pipeline.set_progress_bar_config(disable=True)
    return pipeline


def count_pngs(directory):
    return len(list(Path(directory).glob("*.png"))) if Path(directory).exists() else 0


def generate_images_to_dir(
    pipeline, prompt, out_dir, n, inference_steps, guidance_scale, seed, resolution=512
):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    n_existing = count_pngs(out_dir)
    if n_existing >= n:
        print(f"    {n_existing} immagini già presenti, skip")
        return

    print(f"    {n_existing} già presenti, genero {n - n_existing} mancanti")
    generator = torch.Generator("cuda").manual_seed(seed + n_existing)
    for index in range(n_existing, n):
        image = pipeline(
            prompt,
            num_inference_steps=inference_steps,
            guidance_scale=guidance_scale,
            height=resolution,
            width=resolution,
            generator=generator,
        ).images[0]
        image.save(out_dir / f"gen_{index:04d}.png")


def compute_prdc_metrics(real_features, fake_features, nearest_k=PRDC_NEAREST_K):
    """Calcola PRDC riutilizzando le feature Inception già estratte per FID."""
    metrics = compute_prdc(
        real_features=real_features,
        fake_features=fake_features,
        nearest_k=nearest_k,
    )
    return {
        name: round(float(metrics[name]), 4)
        for name in ("precision", "recall", "density", "coverage")
    }


def eval_one_checkpoint(
    step,
    ckpt_path,
    base_model_dir,
    real_neg_dir,
    real_pos_dir,
    eval_dir,
    n_gen,
    inference_steps,
    guidance_scale,
    seed,
    resolution=512,
):
    class_config = {
        "negative": (NEGATIVE_PROMPT, Path(real_neg_dir), seed),
        "positive": (POSITIVE_PROMPT, Path(real_pos_dir), seed + 1),
    }
    generated_dirs = {
        name: Path(eval_dir) / f"checkpoint-{step}" / name
        for name in class_config
    }

    print(f"\nCheckpoint {step} | {Path(ckpt_path).name}")
    needs_generation = any(count_pngs(path) < n_gen for path in generated_dirs.values())
    pipeline = load_pipeline_from_checkpoint(ckpt_path, base_model_dir) if needs_generation else None
    try:
        for name, (prompt, _, class_seed) in class_config.items():
            if count_pngs(generated_dirs[name]) < n_gen:
                print(f"  Generazione {name}...")
                generate_images_to_dir(
                    pipeline,
                    prompt,
                    generated_dirs[name],
                    n_gen,
                    inference_steps,
                    guidance_scale,
                    class_seed,
                    resolution,
                )
    finally:
        if pipeline is not None:
            del pipeline
            gc.collect()
            torch.cuda.empty_cache()

    metrics = {}
    for name, (_, real_dir, _) in class_config.items():
        print(f"  FID + IS + PRDC: {name}")
        evaluator = GenerativeEvaluator(
            real_dir=real_dir,
            generated_dir=generated_dirs[name],
            batch_size=8,
            num_workers=0,
        )
        fid_is_metrics, real_features, fake_features = evaluator.compute_with_features()
        metrics[name] = {
            **fid_is_metrics,
            **compute_prdc_metrics(real_features, fake_features),
        }

    averages = {
        metric: round(sum(metrics[name][metric] for name in metrics) / len(metrics), 4)
        for metric in ("FID", "IS_mean", "IS_std", "precision", "recall", "density", "coverage")
    }
    print(
        f"  Avg FID={averages['FID']:.4f} | "
        f"IS={averages['IS_mean']:.4f}±{averages['IS_std']:.4f} | "
        f"P={averages['precision']:.4f} | R={averages['recall']:.4f}"
    )
    return {
        "step": step,
        "ckpt_name": Path(ckpt_path).name,
        **metrics,
        "avg_FID": averages["FID"],
        "avg_IS_mean": averages["IS_mean"],
        "avg_IS_std": averages["IS_std"],
        "avg_precision": averages["precision"],
        "avg_recall": averages["recall"],
        "avg_density": averages["density"],
        "avg_coverage": averages["coverage"],
    }


def copy_eval_images_to_final(eval_src_dir, final_dst_dir, max_images, prefix):
    final_dst_dir = Path(final_dst_dir)
    final_dst_dir.mkdir(parents=True, exist_ok=True)
    eval_images = sorted(Path(eval_src_dir).glob("*.png"))[:max_images]
    for index, source_path in enumerate(eval_images):
        shutil.copy2(source_path, final_dst_dir / f"{prefix}_{index:04d}.png")
    print(f"  Riutilizzate {len(eval_images)} immagini da {Path(eval_src_dir).name}.")
    return len(eval_images)


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def remove_generated_duplicates_of_reused_images(final_dir, reused_prefix):
    final_dir = Path(final_dir)
    reused_hashes = {file_sha256(path) for path in final_dir.glob(f"{reused_prefix}_*.png")}
    removed = []
    for path in sorted(final_dir.glob("gen_*.png")):
        if file_sha256(path) in reused_hashes:
            path.unlink()
            removed.append(path)
    if removed:
        print(f"  Rimosse {len(removed)} copie gen_* sovrapposte alle immagini riusate.")
    return removed


def duplicate_png_groups(directory):
    by_hash = {}
    for path in sorted(Path(directory).glob("*.png")):
        by_hash.setdefault(file_sha256(path), []).append(path.name)
    return {digest: names for digest, names in by_hash.items() if len(names) > 1}

In [ ]:
# Valutazione di tutti i checkpoint usando esclusivamente il validation
checkpoints = discover_checkpoints(SD_OUTPUT_DIR)
print(f"Trovati {len(checkpoints)} checkpoint:")
for _step, _path in checkpoints:
    print(f"  checkpoint-{_step}")

expected_checkpoints = {path.name for _, path in checkpoints}
all_metrics = None
if EVAL_METRICS_PATH.is_file():
    with EVAL_METRICS_PATH.open(encoding="utf-8") as handle:
        cached_metrics = json.load(handle)
    cached_checkpoints = {row.get("ckpt_name") for row in cached_metrics}
    required_class_metrics = {
        "FID", "IS_mean", "IS_std", "precision", "recall", "density", "coverage"
    }
    required_average_metrics = {
        "avg_FID", "avg_IS_mean", "avg_IS_std", "avg_precision",
        "avg_recall", "avg_density", "avg_coverage",
    }
    cache_has_complete_metrics = all(
        required_average_metrics <= set(row)
        and all(required_class_metrics <= set(row.get(name, {})) for name in FINAL_GENERATE_CLASSES)
        for row in cached_metrics
    )
    if cached_checkpoints == expected_checkpoints and cache_has_complete_metrics:
        all_metrics = cached_metrics
        print("Metriche validation già complete: riuso il JSON esistente.")
    elif cached_checkpoints == expected_checkpoints:
        print("Il JSON esistente non contiene PRDC: le metriche saranno ricalcolate.")

if all_metrics is None:
    all_metrics = []
    with temporary_real_reference_dirs_from_split_metadata(
        metadata_path=VALIDATION_METADATA_PATH,
        n_per_class=N_VALIDATION_IMAGES_PER_CLASS,
        seed=EVAL_SEED,
    ) as (real_neg_ref_dir, real_pos_ref_dir):
        print("\nDirectory temporanee validation:")
        print("  Negative:", real_neg_ref_dir)
        print("  Positive:", real_pos_ref_dir)

        with measure_sustainability(label="checkpoint_validation_evaluation", sample_interval=0.5) as eco_eval:
            for step, ckpt_path in checkpoints:
                all_metrics.append(eval_one_checkpoint(
                    step=step,
                    ckpt_path=ckpt_path,
                    base_model_dir=LOCAL_MODEL_DIR,
                    real_neg_dir=real_neg_ref_dir,
                    real_pos_dir=real_pos_ref_dir,
                    eval_dir=EVAL_DIR,
                    n_gen=N_EVAL_IMAGES_PER_CLASS,
                    inference_steps=INFERENCE_STEPS,
                    guidance_scale=EVAL_GUIDANCE_SCALE,
                    seed=EVAL_SEED,
                    resolution=RESOLUTION,
                ))

    with EVAL_METRICS_PATH.open("w", encoding="utf-8") as handle:
        json.dump(all_metrics, handle, indent=2, ensure_ascii=False)

    eco_record = eco_eval.metrics.to_dict()
    eco_record.update({
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "record_type": "checkpoint_validation_evaluation",
        "n_checkpoints": len(checkpoints),
        "n_generated_images_per_class": N_EVAL_IMAGES_PER_CLASS,
        "n_real_images_per_class": N_VALIDATION_IMAGES_PER_CLASS,
        "real_reference_metadata": str(VALIDATION_METADATA_PATH),
    })
    with EVAL_SUSTAINABILITY_LOG.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(eco_record, ensure_ascii=False) + "\n")

    print(f"\nMetriche eco-tracking validation:\n{eco_eval.metrics}")
    print(f"Metriche validation salvate in: {EVAL_METRICS_PATH}")

In [ ]:
# Selezione best checkpoint
# Ricarica da file (idempotente se si ri-esegue la cella)
with open(EVAL_METRICS_PATH, encoding="utf-8") as f:
    all_metrics = json.load(f)

df_eval = pd.DataFrame([
    {
        "checkpoint":    m["ckpt_name"],
        "step":          m["step"],
        "avg_FID":       m["avg_FID"],
        "avg_IS_mean":   m["avg_IS_mean"],
        "avg_IS_std":    m["avg_IS_std"],
        "avg_precision": m["avg_precision"],
        "avg_recall":    m["avg_recall"],
        "avg_density":   m["avg_density"],
        "avg_coverage":  m["avg_coverage"],
        "FID_neg":       m["negative"]["FID"],
        "FID_pos":       m["positive"]["FID"],
        "IS_neg":        m["negative"]["IS_mean"],
        "IS_pos":        m["positive"]["IS_mean"],
        "precision_neg": m["negative"]["precision"],
        "precision_pos": m["positive"]["precision"],
        "recall_neg":    m["negative"]["recall"],
        "recall_pos":    m["positive"]["recall"],
    }
    for m in all_metrics
]).sort_values("avg_FID").reset_index(drop=True)

print("Riepilogo metriche (ordinato per avg_FID crescente - migliore prima):")
print(df_eval.to_string(index=False))

best_row = df_eval.iloc[0]
BEST_CHECKPOINT = SD_OUTPUT_DIR / best_row["checkpoint"]

print(f"\nMiglior checkpoint : {best_row['checkpoint']}")
print(f"  avg FID       : {best_row['avg_FID']:.4f}")
print(f"  avg IS        : {best_row['avg_IS_mean']:.4f} ± {best_row['avg_IS_std']:.4f}")
print(f"  avg precision : {best_row['avg_precision']:.4f}")
print(f"  avg recall    : {best_row['avg_recall']:.4f}")
print(f"  Path          : {BEST_CHECKPOINT}")

# Grafici diagnostici separati; il FID resta nella figura dedicata della cella successiva.
df_eval_by_step = df_eval.sort_values("step").reset_index(drop=True)
best_step = int(best_row["step"])

figure, axis = plt.subplots(figsize=(10, 5), constrained_layout=True)
axis.plot(
    df_eval_by_step["step"],
    df_eval_by_step["avg_IS_mean"],
    color="#9467bd",
    marker="o",
    label="IS medio",
)
axis.fill_between(
    df_eval_by_step["step"],
    df_eval_by_step["avg_IS_mean"] - df_eval_by_step["avg_IS_std"],
    df_eval_by_step["avg_IS_mean"] + df_eval_by_step["avg_IS_std"],
    color="#9467bd",
    alpha=0.2,
    label="± deviazione standard",
)
axis.axvline(best_step, color="red", linestyle="--", alpha=0.45, label="best FID")
axis.set(title="Inception Score medio per checkpoint", xlabel="Training step", ylabel="IS")
axis.grid(alpha=0.25)
axis.legend()
figure.savefig(PLOTS_DIR / "is_per_checkpoint.png", dpi=180, bbox_inches="tight")
plt.show()
plt.close(figure)

figure, axis = plt.subplots(figsize=(10, 5), constrained_layout=True)
for column, label, color in [
    ("avg_precision", "Precision media", "#1f77b4"),
    ("avg_recall", "Recall media", "#ff7f0e"),
]:
    axis.plot(
        df_eval_by_step["step"],
        df_eval_by_step[column],
        color=color,
        marker="o",
        label=label,
    )
axis.axvline(best_step, color="red", linestyle="--", alpha=0.45, label="best FID")
axis.set(
    title="Precision e recall medie per checkpoint",
    xlabel="Training step",
    ylabel="Valore PRDC",
)
axis.grid(alpha=0.25)
axis.legend()
figure.savefig(PLOTS_DIR / "precision_recall_over_steps.png", dpi=180, bbox_inches="tight")
plt.show()
plt.close(figure)

figure, axis = plt.subplots(figsize=(10, 5), constrained_layout=True)
for column, label, color in [
    ("avg_density", "Density media", "#2ca02c"),
    ("avg_coverage", "Coverage media", "#d62728"),
]:
    axis.plot(
        df_eval_by_step["step"],
        df_eval_by_step[column],
        color=color,
        marker="o",
        label=label,
    )
axis.axvline(best_step, color="red", linestyle="--", alpha=0.45, label="best FID")
axis.set(
    title="Density e coverage medie per checkpoint",
    xlabel="Training step",
    ylabel="Valore PRDC",
)
axis.grid(alpha=0.25)
axis.legend()
figure.savefig(PLOTS_DIR / "density_coverage_over_steps.png", dpi=180, bbox_inches="tight")
plt.show()
plt.close(figure)

print("Grafici diagnostici checkpoint separati salvati in:", PLOTS_DIR)

### Diagnostica della selezione del checkpoint

I grafici successivi trasformano il JSON di valutazione in una lettura metodologica del
percorso di training, senza ricalcolare le metriche. Ogni famiglia di metriche viene
mostrata in una figura separata per evitare sovrapposizioni e duplicazioni:

- FID negative, positive e medio in funzione dello step;
- Inception Score medio con banda di deviazione standard;
- precision e recall medie;
- density e coverage medie;
- scatter precision-recall dei checkpoint.

Il checkpoint scelto viene evidenziato, ma la regola di selezione resta esclusivamente il
minimo `avg_FID`. Le figure vengono mostrate nel notebook e salvate in
`results/03b_finetuning_filtered/plots/`.


In [ ]:
with EVAL_METRICS_PATH.open(encoding="utf-8") as handle:
    checkpoint_plot_metrics = json.load(handle)

df_checkpoint_plots = pd.DataFrame([
    {
        "checkpoint": row["ckpt_name"],
        "step": row["step"],
        "FID negative": row["negative"]["FID"],
        "FID positive": row["positive"]["FID"],
        "FID medio": row["avg_FID"],
        "precision media": row["avg_precision"],
        "recall media": row["avg_recall"],
    }
    for row in checkpoint_plot_metrics
]).sort_values("step")

best_plot_row = df_checkpoint_plots.loc[df_checkpoint_plots["FID medio"].idxmin()]

fig, ax = plt.subplots(figsize=(10, 5))
for column, marker in [
    ("FID negative", "o"),
    ("FID positive", "s"),
    ("FID medio", "^"),
]:
    ax.plot(df_checkpoint_plots["step"], df_checkpoint_plots[column], marker=marker, label=column)
ax.scatter(
    [best_plot_row["step"]],
    [best_plot_row["FID medio"]],
    s=180,
    facecolors="none",
    edgecolors="red",
    linewidths=2,
    label=f"best: {best_plot_row['checkpoint']}",
)
ax.set(title="FID sul validation set per checkpoint", xlabel="Training step", ylabel="FID")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(PLOTS_DIR / "fid_per_checkpoint.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 6))
scatter = ax.scatter(
    df_checkpoint_plots["recall media"],
    df_checkpoint_plots["precision media"],
    c=df_checkpoint_plots["step"],
    cmap="viridis",
    s=65,
)
ax.scatter(
    [best_plot_row["recall media"]],
    [best_plot_row["precision media"]],
    s=220,
    facecolors="none",
    edgecolors="red",
    linewidths=2,
    label=f"best FID: {best_plot_row['checkpoint']}",
)
ax.set(
    title="PRDC medio per checkpoint",
    xlabel="Recall medio",
    ylabel="Precision media",
)
ax.grid(alpha=0.25)
ax.legend()
fig.colorbar(scatter, ax=ax, label="Training step")
fig.tight_layout()
fig.savefig(PLOTS_DIR / "precision_recall_per_checkpoint.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

print("Grafici checkpoint salvati in:", PLOTS_DIR)

### Curva della loss di training

La cella seguente cerca gli eventi TensorBoard prodotti dal fine-tuning e, quando
disponibili, ricostruisce la curva della loss. Nel training diffusion questa loss misura
principalmente l'errore di predizione del rumore: è utile per verificare stabilità,
convergenza e anomalie, ma **non è una misura diretta della qualità percettiva** delle
immagini generate e può correlare debolmente con essa.

Per confrontare la qualità dei checkpoint si devono quindi usare soprattutto FID e le
metriche PRDC sul validation set. Se non esistono eventi o tag di loss compatibili, la
cella lo segnala e termina senza errore.


In [ ]:
from tensorboard.backend.event_processing import event_accumulator

event_files = sorted((SD_OUTPUT_DIR / "logs").rglob("events.out.tfevents.*"))
loss_rows = []

for event_file in event_files:
    try:
        accumulator = event_accumulator.EventAccumulator(
            str(event_file),
            size_guidance={"scalars": 0},
        )
        accumulator.Reload()
        scalar_tags = accumulator.Tags().get("scalars", [])
        preferred_tags = ["train_loss", "loss"]
        loss_tag = next((tag for tag in preferred_tags if tag in scalar_tags), None)
        if loss_tag is None:
            loss_tag = next((tag for tag in scalar_tags if tag.endswith("/loss")), None)
        if loss_tag is None:
            continue
        loss_rows.extend({
            "step": event.step,
            "loss": event.value,
            "wall_time": event.wall_time,
            "source": event_file.name,
            "tag": loss_tag,
        } for event in accumulator.Scalars(loss_tag))
    except Exception as exc:
        print(f"Evento TensorBoard non leggibile ({event_file.name}): {exc}")

if not loss_rows:
    print("Nessuna serie di loss TensorBoard disponibile: grafico non generato.")
else:
    df_loss = (
        pd.DataFrame(loss_rows)
        .sort_values(["step", "wall_time"])
        .drop_duplicates(subset="step", keep="last")
    )
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(df_loss["step"], df_loss["loss"], linewidth=1.2)
    ax.set(
        title="Loss di training registrata da TensorBoard",
        xlabel="Training step",
        ylabel="Loss",
    )
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(PLOTS_DIR / "train_loss.png", dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("Curva della loss salvata in:", PLOTS_DIR / "train_loss.png")

## 6. Generazione RAW finale a 100 inference step

La cella successiva usa il checkpoint selezionato sul validation set per costruire il
dataset sintetico RAW finale.

Per ciascuna classe (`positive` e `negative`) vengono prodotte esattamente 2722 immagini (1361 sarebbe il numero necessario per bilanciare positivi e negativi nel train set, ne generiamo il doppio) in `experiments/.../generated_images/final/<class_name>/`. Le prime immagini disponibili dalla valutazione del checkpoint migliore vengono copiate e riutilizzate; la pipeline genera soltanto quelle mancanti.

Prima e dopo la generazione vengono effettuati controlli sul numero di file e sui duplicati byte-per-byte. Se entrambe le classi sono già complete, la pipeline non viene caricata e l'operazione viene saltata. Un eventuale cambio del checkpoint migliore in presenza di immagini già generate viene segnalato per evitare di mescolare output di modelli diversi.

Il riepilogo della generazione viene aggiornato in `results/03b_finetuning_filtered/metrics/generation_info.json`, mentre le misure ambientali vengono aggiunte a `results/03b_finetuning_filtered/ecotracker/sustainability_generation.jsonl`.


In [ ]:
# Generazione finale con il miglior checkpoint
print(f"Best checkpoint  : {BEST_CHECKPOINT.name}")
print(f"Immagini/classe  : {N_FINAL_IMAGES_PER_CLASS}")
print(f"Output           : {FINAL_GEN_DIR}")

BEST_EVAL_DIR = EVAL_DIR / BEST_CHECKPOINT.name
_class_config = {
    "negative": {
        "prompt": NEGATIVE_PROMPT,
        "final_dir": FINAL_NEG_DIR,
        "eval_dir": BEST_EVAL_DIR / "negative",
        "seed": EVAL_SEED,
        "prefix": "eval_neg",
    },
    "positive": {
        "prompt": POSITIVE_PROMPT,
        "final_dir": FINAL_POS_DIR,
        "eval_dir": BEST_EVAL_DIR / "positive",
        "seed": EVAL_SEED + 1,
        "prefix": "eval_pos",
    },
}

_final_already_complete = all(
    count_pngs(_class_config[_cls]["final_dir"]) == N_FINAL_IMAGES_PER_CLASS
    for _cls in FINAL_GENERATE_CLASSES
)
if _final_already_complete:
    print("Dataset finale già completo: la generazione verrà saltata.")
generation_info_path = GENERATION_INFO_PATH
if generation_info_path.exists():
    with open(generation_info_path, encoding="utf-8") as f:
        previous_generation = json.load(f)
    previous_best = previous_generation.get("best_checkpoint")
    final_images_exist = any(count_pngs(cfg["final_dir"]) > 0 for cfg in _class_config.values())
    if previous_best and previous_best != BEST_CHECKPOINT.name and final_images_exist:
        raise RuntimeError(
            f"Il best checkpoint e' cambiato da {previous_best} a {BEST_CHECKPOINT.name}. "
            "Usa una nuova FINAL_GEN_DIR o svuota consapevolmente le cartelle finali."
        )

pipe_final = None if _final_already_complete else load_pipeline_from_checkpoint(BEST_CHECKPOINT, LOCAL_MODEL_DIR)
removed_overlaps = {}

with measure_sustainability(label=f"final_gen_{BEST_CHECKPOINT.name}", sample_interval=0.5) as eco_final:
    for _cls in FINAL_GENERATE_CLASSES:
        cfg = _class_config[_cls]
        if count_pngs(cfg["final_dir"]) == N_FINAL_IMAGES_PER_CLASS:
            print(f"Classe {_cls.upper()}: {N_FINAL_IMAGES_PER_CLASS} immagini già presenti, skip")
            continue
        eval_available = min(count_pngs(cfg["eval_dir"]), N_EVAL_IMAGES_PER_CLASS)

        print(f"\nClasse {_cls.upper()}")
        print(f"  Target finale              : {N_FINAL_IMAGES_PER_CLASS}")
        print(f"  Immagini eval riutilizzate : {eval_available}")

        # Copia prima le immagini riusate, poi completa il totale.
        copy_eval_images_to_final(
            eval_src_dir=cfg["eval_dir"],
            final_dst_dir=cfg["final_dir"],
            max_images=eval_available,
            prefix=cfg["prefix"],
        )
        removed = remove_generated_duplicates_of_reused_images(
            final_dir=cfg["final_dir"],
            reused_prefix=cfg["prefix"],
        )
        removed_overlaps[_cls] = len(removed)

        n_before = count_pngs(cfg["final_dir"])
        print(f"  Immagini uniche presenti   : {n_before}")
        print(f"  Nuove da generare          : {max(0, N_FINAL_IMAGES_PER_CLASS - n_before)}")

        generate_images_to_dir(
            pipe_final,
            cfg["prompt"],
            cfg["final_dir"],
            N_FINAL_IMAGES_PER_CLASS,
            INFERENCE_STEPS,
            EVAL_GUIDANCE_SCALE,
            cfg["seed"],
            RESOLUTION,
        )

        duplicates = duplicate_png_groups(cfg["final_dir"])
        if duplicates:
            raise RuntimeError(f"Trovati {len(duplicates)} gruppi duplicati in {cfg['final_dir']}.")
        if count_pngs(cfg["final_dir"]) != N_FINAL_IMAGES_PER_CLASS:
            raise RuntimeError(f"Numero finale non valido per {_cls}.")

del pipe_final
gc.collect()
torch.cuda.empty_cache()

_n_neg = count_pngs(FINAL_NEG_DIR)
_n_pos = count_pngs(FINAL_POS_DIR)
_final_record = eco_final.metrics.to_dict()
_final_record.update({
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "record_type": "final_generation",
    "generation_log_schema": 2,
    "best_checkpoint": BEST_CHECKPOINT.name,
    "selection_reference": "validation",
    "selection_metrics_path": str(EVAL_METRICS_PATH),
    "avg_FID": float(best_row["avg_FID"]),
    "avg_IS_mean": float(best_row["avg_IS_mean"]),
    "n_per_class": N_FINAL_IMAGES_PER_CLASS,
    "n_eval_reused": N_EVAL_IMAGES_PER_CLASS,
    "removed_overlaps": removed_overlaps,
    "n_negative_final": _n_neg,
    "n_positive_final": _n_pos,
    "inference_steps": INFERENCE_STEPS,
    "guidance_scale": EVAL_GUIDANCE_SCALE,
    "generated_classes": FINAL_GENERATE_CLASSES,
    "generated_negative": str(FINAL_NEG_DIR),
    "generated_positive": str(FINAL_POS_DIR),
    "status": "completed",
})

with open(generation_info_path, "w", encoding="utf-8") as f:
    json.dump(_final_record, f, indent=2, ensure_ascii=False)
with open(FINAL_GENERATION_LOG, "a", encoding="utf-8") as f:
    f.write(json.dumps(_final_record, ensure_ascii=False) + "\n")

print(f"\nMetriche eco-tracking generazione finale:\n{eco_final.metrics}")
print(f"Positive finali: {_n_pos} -> {FINAL_POS_DIR}")

## 7. Filtro adattivo non supervisionato per entrambe le classi

Le due celle successive definiscono e applicano un filtro di qualità basato unicamente su proprietà dell'immagine, senza classificatori e senza consultare validation o test per calibrare le soglie.

Per ogni immagine viene stimata una foreground mask adattiva a partire da bordi e angoli. Sulla maschera vengono calcolate statistiche quali area non nera, intensità, contrasto, entropia, varianza del Laplaciano e compattezza della componente principale.

La calibrazione viene eseguita **separatamente per classe** usando esclusivamente le immagini reali del train della stessa label. Il filtro procede poi in due fasi:

1. applica regole dure per rifiutare immagini troppo vuote, troppo piene, poco contrastate o con foreground frammentato;
2. ordina le immagini accettate in base alla distanza robusta dalle statistiche del train e seleziona le migliori 1361 per classe.

Le immagini RAW non vengono modificate. Le selezionate sono copiate nelle directory `<class_name>_filtered_1361_adaptive_mask`; report CSV e summary JSON documentano soglie, score, motivi di rifiuto e numerosità. Se una directory filtrata è già completa e priva di duplicati, la relativa classe viene saltata.

Questo filtro privilegia immagini regolari e vicine alla distribuzione reale; il confronto successivo serve anche a verificare l'eventuale perdita di diversità causata dalla selezione.


In [ ]:
import numpy as np
from scipy import ndimage as ndi

QUALITY_FEATURES = [
    "nonblack_ratio",
    "mean_nonblack",
    "std_nonblack",
    "entropy",
    "lap_var",
    "largest_component_frac",
]


def estimate_foreground_mask(arr, min_threshold=10):
    arr = np.asarray(arr, dtype=np.uint8)
    height, width = arr.shape
    border_width = max(2, int(round(min(height, width) * 0.02)))
    corner_height = max(8, int(round(height * 0.12)))
    corner_width = max(8, int(round(width * 0.12)))

    background_candidates = np.concatenate([
        arr[:border_width, :].ravel(),
        arr[-border_width:, :].ravel(),
        arr[:, :border_width].ravel(),
        arr[:, -border_width:].ravel(),
        arr[:corner_height, :corner_width].ravel(),
        arr[:corner_height, -corner_width:].ravel(),
        arr[-corner_height:, :corner_width].ravel(),
        arr[-corner_height:, -corner_width:].ravel(),
    ]).astype(np.float64)

    low_background = background_candidates[
        background_candidates <= np.quantile(background_candidates, 0.60)
    ]
    background_median = float(np.median(low_background))
    background_mad = float(np.median(np.abs(low_background - background_median)))
    adaptive_threshold = max(
        float(min_threshold),
        background_median + 6.0 * 1.4826 * background_mad + 5.0,
    )
    adaptive_threshold = max(
        float(min_threshold),
        min(adaptive_threshold, 220.0, float(np.quantile(arr, 0.95))),
    )

    raw_mask = arr > adaptive_threshold
    raw_labels, component_count = ndi.label(raw_mask)
    component_sizes = (
        np.bincount(raw_labels.ravel())[1:]
        if component_count
        else np.array([], dtype=np.int64)
    )
    largest_component_frac = (
        float(component_sizes.max() / max(int(raw_mask.sum()), 1))
        if component_sizes.size
        else 0.0
    )

    structure = ndi.generate_binary_structure(2, 1)
    clean_mask = ndi.binary_opening(raw_mask, structure=structure, iterations=1)
    clean_mask = ndi.binary_closing(clean_mask, structure=structure, iterations=2)
    clean_mask = ndi.binary_fill_holes(clean_mask)
    clean_labels, clean_count = ndi.label(clean_mask)
    clean_sizes = (
        np.bincount(clean_labels.ravel())[1:]
        if clean_count
        else np.array([], dtype=np.int64)
    )
    if clean_sizes.size:
        clean_mask = clean_labels == int(clean_sizes.argmax() + 1)
    else:
        clean_mask = np.zeros_like(raw_mask, dtype=bool)

    diagnostics = {
        "adaptive_threshold": float(adaptive_threshold),
        "background_median": background_median,
        "background_mad": background_mad,
        "raw_nonblack_ratio": float(raw_mask.mean()),
        "largest_component_frac": largest_component_frac,
    }
    return clean_mask, diagnostics


def image_quality_metrics(path, nonblack_threshold=NONBLACK_THRESHOLD):
    with Image.open(path) as image:
        gray = np.asarray(image.convert("L"), dtype=np.uint8)

    mask, diagnostics = estimate_foreground_mask(gray, nonblack_threshold)
    foreground_values = gray[mask].astype(np.float64)
    foreground_count = int(mask.sum())
    pixel_count = int(mask.size)

    histogram = np.bincount(gray.ravel(), minlength=256).astype(np.float64)
    probabilities = histogram[histogram > 0] / pixel_count
    entropy = float(-(probabilities * np.log2(probabilities)).sum())

    return {
        **diagnostics,
        "nonblack_ratio": float(foreground_count / pixel_count),
        "mean_nonblack": float(foreground_values.mean()) if foreground_count else 0.0,
        "std_nonblack": float(foreground_values.std()) if foreground_count else 0.0,
        "entropy": entropy,
        "lap_var": float(ndi.laplace(gray.astype(np.float32)).var()),
        "top_border": float(mask[0, :].mean()),
        "bottom_border": float(mask[-1, :].mean()),
    }


def compute_reference_thresholds(label: int):
    train_paths = get_real_image_paths_from_split_metadata(
        metadata_path=TRAIN_METADATA_PATH,
        label=label,
        n_images=None,
        seed=EVAL_SEED + label,
    )
    rows = []
    for index, path in enumerate(train_paths, start=1):
        rows.append({"path": str(path), **image_quality_metrics(path)})
        if index % 100 == 0 or index == len(train_paths):
            print(f"  Calibrazione label={label}: {index}/{len(train_paths)}")

    train_metrics = pd.DataFrame(rows)
    thresholds = {
        "min_nonblack_ratio": float(max(0.02, train_metrics["nonblack_ratio"].quantile(0.01) * 0.70)),
        "max_nonblack_ratio": float(min(0.95, train_metrics["nonblack_ratio"].quantile(0.99) * 1.30)),
        "min_std_nonblack": float(max(3.0, train_metrics["std_nonblack"].quantile(0.01) * 0.50)),
        "min_largest_component_frac": 0.75,
        "max_top_border": 0.65,
        "max_bottom_border": 0.65,
    }
    medians = {
        feature: float(train_metrics[feature].median())
        for feature in QUALITY_FEATURES
    }
    iqrs = {}
    for feature in QUALITY_FEATURES:
        iqr = float(
            train_metrics[feature].quantile(0.75)
            - train_metrics[feature].quantile(0.25)
        )
        iqrs[feature] = iqr if iqr > 0 else 1e-6

    return {
        "thresholds": thresholds,
        "medians": medians,
        "iqrs": iqrs,
        "n_train": len(train_paths),
        "train_metrics": train_metrics,
    }


def score_candidate(row, reference):
    thresholds = reference["thresholds"]
    rejection_rules = [
        (row["nonblack_ratio"] < thresholds["min_nonblack_ratio"], "too_empty"),
        (row["nonblack_ratio"] > thresholds["max_nonblack_ratio"], "too_full"),
        (row["std_nonblack"] < thresholds["min_std_nonblack"], "low_contrast"),
        (
            row["largest_component_frac"] < thresholds["min_largest_component_frac"],
            "fragmented_mask",
        ),
        (row["top_border"] > thresholds["max_top_border"], "touches_top_border"),
        (row["bottom_border"] > thresholds["max_bottom_border"], "touches_bottom_border"),
    ]
    for rejected, reason in rejection_rules:
        if rejected:
            return True, reason, np.nan

    selection_score = -sum(
        abs(float(row[feature]) - reference["medians"][feature])
        / reference["iqrs"][feature]
        for feature in QUALITY_FEATURES
    )
    return False, "", float(selection_score)


def evaluate_generated_dir_against_split(generated_dir, metadata_path, label, n_images, seed):
    generated_dir = Path(generated_dir)
    n_generated = count_pngs(generated_dir)
    if n_generated == 0:
        raise RuntimeError(f"Nessuna immagine generata in {generated_dir}.")
    duplicates = duplicate_png_groups(generated_dir)
    if duplicates:
        raise RuntimeError(f"Trovati {len(duplicates)} gruppi duplicati in {generated_dir}.")

    with temporary_real_reference_dir_from_split_metadata(
        metadata_path=metadata_path,
        label=label,
        n_images=n_images,
        seed=seed,
    ) as real_reference_dir:
        evaluator = GenerativeEvaluator(
            real_dir=real_reference_dir,
            generated_dir=generated_dir,
            batch_size=8,
            num_workers=0,
        )
        fid_is_metrics, real_features, fake_features = evaluator.compute_with_features()
        prdc_metrics = compute_prdc_metrics(real_features, fake_features)

    return {
        **fid_is_metrics,
        **prdc_metrics,
        "n_generated": n_generated,
    }

In [ ]:
# Filtra entrambe le classi con calibrazione indipendente.
filter_summaries = {}

for class_name in FINAL_GENERATE_CLASSES:
    class_label = FINAL_CLASS_LABELS[class_name]
    raw_dir = FINAL_DIRS[class_name]
    filtered_dir = FILTERED_DIRS[class_name]
    report_path = FILTER_REPORT_PATHS[class_name]
    summary_path = FILTER_SUMMARY_PATHS[class_name]

    if count_pngs(raw_dir) != N_FINAL_IMAGES_PER_CLASS:
        raise RuntimeError(
            f"Dataset RAW incompleto per {class_name}: "
            f"{count_pngs(raw_dir)} != {N_FINAL_IMAGES_PER_CLASS}."
        )

    if count_pngs(filtered_dir) == N_SELECTED_PER_CLASS:
        if duplicate_png_groups(filtered_dir):
            raise RuntimeError(f"Il dataset filtrato {class_name} contiene duplicati.")
        print(f"{class_name}: {N_SELECTED_PER_CLASS} immagini filtrate già presenti, skip")
        filter_summaries[class_name] = {"status": "already_complete"}
        continue

    print(f"\nCalibrazione filtro per {class_name} (label={class_label})...")
    reference_thresholds = compute_reference_thresholds(class_label)
    candidate_rows = []
    candidate_paths = sorted(raw_dir.glob("*.png"))
    for index, path in enumerate(candidate_paths, start=1):
        metrics = image_quality_metrics(path)
        rejected, reject_reason, selection_score = score_candidate(
            metrics,
            reference_thresholds,
        )
        candidate_rows.append({
            "class": class_name,
            "source_path": str(path),
            "source_name": path.name,
            **metrics,
            "rejected": rejected,
            "reject_reason": reject_reason,
            "selection_score": selection_score,
        })
        if index % 250 == 0 or index == len(candidate_paths):
            print(f"  Analisi {class_name}: {index}/{len(candidate_paths)}")

    candidates = pd.DataFrame(candidate_rows)
    accepted = (
        candidates.loc[~candidates["rejected"]]
        .sort_values("selection_score", ascending=False)
        .copy()
    )
    reject_counts = {
        str(reason): int(count)
        for reason, count in candidates.loc[
            candidates["rejected"], "reject_reason"
        ].value_counts().items()
    }

    if len(accepted) < N_SELECTED_PER_CLASS:
        candidates.to_csv(report_path, index=False)
        raise RuntimeError(
            f"Dopo il filtro {class_name} restano {len(accepted)} immagini; "
            f"ne servono {N_SELECTED_PER_CLASS}."
        )

    selected = accepted.head(N_SELECTED_PER_CLASS).copy()
    selected["selection_rank"] = range(N_SELECTED_PER_CLASS)
    rank_by_path = dict(zip(selected["source_path"], selected["selection_rank"]))
    candidates["selected"] = candidates["source_path"].isin(rank_by_path)
    candidates["selection_rank"] = candidates["source_path"].map(rank_by_path)

    for path in filtered_dir.glob("*.png"):
        path.unlink()
    for rank, source_path in enumerate(selected["source_path"]):
        shutil.copy2(
            source_path,
            filtered_dir / f"selected_{class_name}_{rank:04d}.png",
        )

    if count_pngs(filtered_dir) != N_SELECTED_PER_CLASS:
        raise RuntimeError(f"Numero immagini filtrate non valido per {class_name}.")
    if duplicate_png_groups(filtered_dir):
        raise RuntimeError(f"Il dataset filtrato {class_name} contiene duplicati.")

    candidates.to_csv(report_path, index=False)
    summary = {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "experiment": EXPERIMENT_NAME,
        "class": class_name,
        "label": class_label,
        "raw_dir": str(raw_dir),
        "filtered_dir": str(filtered_dir),
        "n_raw": len(candidates),
        "n_accepted": len(accepted),
        "n_rejected": int(candidates["rejected"].sum()),
        "n_selected": N_SELECTED_PER_CLASS,
        "thresholds": reference_thresholds["thresholds"],
        "reference_medians": reference_thresholds["medians"],
        "reference_iqrs": reference_thresholds["iqrs"],
        "n_real_train_reference": reference_thresholds["n_train"],
        "reject_reason_counts": reject_counts,
        "status": "completed",
    }
    with summary_path.open("w", encoding="utf-8") as handle:
        json.dump(summary, handle, indent=2, ensure_ascii=False)
    filter_summaries[class_name] = summary

    print(f"{class_name}: selezionate {N_SELECTED_PER_CLASS} immagini")
    print("Report:", report_path)
    print("Summary:", summary_path)

## 8. Confronto RAW vs filtrate sul validation set

La cella successiva misura l'effetto del filtro senza coinvolgere il test set.

Per ciascuna classe vengono valutati contro i reali del validation set:

- il dataset RAW completo da 2722 immagini;
- il dataset filtrato da 1361 immagini selezionate con adaptive mask.

Per ogni combinazione vengono calcolati FID, Inception Score, precision, recall, density e coverage, oltre al numero di immagini generate. I risultati confluiscono in un unico DataFrame, rendendo esplicito se il filtro migliori la fedeltà a costo di ridurre la copertura della distribuzione reale.

Il confronto viene salvato in `results/03b_finetuning_filtered/metrics/validation_comparison_raw_vs_filtered.csv` e `results/03b_finetuning_filtered/metrics/validation_comparison_raw_vs_filtered.json`. Un FID più basso indica maggiore vicinanza complessiva ai reali. Precision e density aiutano a leggere la fedeltà, mentre recall e coverage evidenziano eventuali perdite di varietà introdotte dalla selezione. Le metriche vanno quindi interpretate congiuntamente.


In [ ]:
comparison_rows = []
for class_name in FINAL_GENERATE_CLASSES:
    label = FINAL_CLASS_LABELS[class_name]
    sets_to_evaluate = {
        f"{class_name}_raw_{N_FINAL_IMAGES_PER_CLASS}": FINAL_DIRS[class_name],
        f"{class_name}_filtered_{N_SELECTED_PER_CLASS}_adaptive_mask": FILTERED_DIRS[class_name],
    }
    for set_name, generated_dir in sets_to_evaluate.items():
        print(f"Valutazione validation: {set_name}")
        metrics = evaluate_generated_dir_against_split(
            generated_dir=generated_dir,
            metadata_path=VALIDATION_METADATA_PATH,
            label=label,
            n_images=N_VALIDATION_IMAGES_PER_CLASS,
            seed=EVAL_SEED + label,
        )
        comparison_rows.append({
            "class": class_name,
            "set_name": set_name,
            **metrics,
        })

metric_columns = [
    "FID", "IS_mean", "IS_std", "precision", "recall", "density", "coverage"
]
df_validation_comparison = pd.DataFrame(comparison_rows)[
    ["class", "set_name", *metric_columns, "n_generated"]
]
df_validation_comparison.to_csv(VALIDATION_COMPARISON_CSV, index=False)
with VALIDATION_COMPARISON_JSON.open("w", encoding="utf-8") as handle:
    json.dump({
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "experiment": EXPERIMENT_NAME,
        "evaluation_reference": "validation",
        "metrics": df_validation_comparison.to_dict(orient="records"),
    }, handle, indent=2, ensure_ascii=False)

print("\nConfronto RAW vs filtrate su validation:")
print(df_validation_comparison.to_string(index=False))
print("\nSalvato in:", VALIDATION_COMPARISON_CSV)

### Visualizzazione dell'effetto del filtro

I grafici successivi leggono il confronto salvato su validation e rendono visibile il
compromesso introdotto dal filtro adattivo. Il FID confronta la distanza complessiva dai
reali; precision e recall separano invece fedeltà e copertura. Un aumento della precision
accompagnato da una riduzione del recall può indicare che il filtro rimuove artefatti ma
restringe anche la varietà delle immagini.

Infine vengono mostrate due griglie indipendenti di campioni filtrati, una positiva e una
negativa. Le griglie servono come controllo qualitativo complementare alle metriche e
vengono salvate insieme agli altri grafici in `results/03b_finetuning_filtered/plots/`.


In [ ]:
df_validation_filter_plots = pd.read_csv(VALIDATION_COMPARISON_CSV)
df_validation_filter_plots["variant"] = df_validation_filter_plots["set_name"].map(
    lambda name: "Filtrate" if "_filtered_" in name else "RAW"
)

fid_pivot = (
    df_validation_filter_plots.pivot(index="class", columns="variant", values="FID")
    .reindex(index=FINAL_GENERATE_CLASSES, columns=["RAW", "Filtrate"])
)
fig, ax = plt.subplots(figsize=(8, 5))
fid_pivot.plot(kind="bar", ax=ax, rot=0)
ax.set(title="FID RAW vs filtrate sul validation set", xlabel="Classe", ylabel="FID")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(PLOTS_DIR / "raw_vs_filtered_fid.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True)
for ax, metric, title in [
    (axes[0], "precision", "Precision PRDC"),
    (axes[1], "recall", "Recall PRDC"),
]:
    pivot = (
        df_validation_filter_plots.pivot(index="class", columns="variant", values=metric)
        .reindex(index=FINAL_GENERATE_CLASSES, columns=["RAW", "Filtrate"])
    )
    pivot.plot(kind="bar", ax=ax, rot=0)
    ax.set(title=title, xlabel="Classe", ylabel=metric.capitalize())
    ax.grid(axis="y", alpha=0.25)
fig.suptitle("PRDC RAW vs filtrate sul validation set")
fig.tight_layout()
fig.savefig(PLOTS_DIR / "raw_vs_filtered_precision_recall.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

for class_name in FINAL_GENERATE_CLASSES:
    sample_dir = FILTERED_DIRS[class_name]
    image_paths = sorted(sample_dir.glob("*.png"))
    if not image_paths:
        print(f"Nessuna immagine disponibile per la griglia {class_name}.")
        continue

    n_samples = min(16, len(image_paths))
    sample_indices = np.linspace(0, len(image_paths) - 1, n_samples, dtype=int)
    selected_paths = [image_paths[index] for index in sample_indices]

    fig, axes = plt.subplots(4, 4, figsize=(10, 10))
    for ax, image_path in zip(axes.flat, selected_paths):
        with Image.open(image_path) as image:
            ax.imshow(image.convert("L"), cmap="gray")
        ax.set_title(image_path.name, fontsize=7)
        ax.axis("off")
    for ax in axes.flat[n_samples:]:
        ax.axis("off")
    fig.suptitle(f"Campioni generati filtrati - {class_name}")
    fig.tight_layout()
    output_path = PLOTS_DIR / f"samples_{class_name}.png"
    fig.savefig(output_path, dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)

print("Grafici del confronto e griglie salvati in:", PLOTS_DIR)

### Trade-off introdotto dal filtro adattivo

Questa sezione usa **esclusivamente artefatti già salvati**, senza ricalcolare metriche e
senza utilizzare la GPU. Il confronto prima/dopo legge
`results/03b_finetuning_filtered/metrics/validation_comparison_raw_vs_filtered.csv`;
il relativo `results/03b_finetuning_filtered/metrics/final_test_metrics.json` viene
letto solo come controllo contestuale sulle immagini filtrate, perché sul test set non è
disponibile un confronto RAW equivalente.

Il messaggio interpretativo da verificare visivamente è il seguente:

> Il filtro adattivo aumenta precision e density, migliorando la fedeltà alla
> distribuzione reale, e riduce il FID, accettando una variazione contenuta della recall.
> Il calo dell'Inception Score descrive quindi un trade-off tra fedeltà e varietà, non una
> perdita automatica di qualità.

Il primo grafico mostra direttamente le traiettorie RAW-filtrate per classe, mantenendo
ogni metrica sul proprio asse. Il radar confronta esclusivamente le quattro metriche PRDC,
che hanno scale compatibili. Le barre divergenti esprimono infine la variazione
percentuale **orientata al miglioramento**: per il FID il segno viene invertito, perché un
valore più basso è migliore. I tre grafici restano separati per mantenerne leggibilità
e ruolo metodologico distinto. Tutti gli output vengono salvati in
`results/03b_finetuning_filtered/plots/`.


In [ ]:
from matplotlib.patches import Patch

# Sorgenti in sola lettura dell'esperimento corrente a 100 inference step.
FILTER_COMPARISON_CSV = VALIDATION_COMPARISON_CSV
FILTER_FINAL_TEST_JSON = FINAL_TEST_METRICS_PATH


def show_and_save_filter_figure(figure, filename):
    output_path = PLOTS_DIR / filename
    figure.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.show()
    plt.close(figure)
    print("Salvato:", output_path)


if not FILTER_COMPARISON_CSV.is_file():
    print(
        "Confronto RAW/filtrate non disponibile: grafici non generati. File atteso:",
        FILTER_COMPARISON_CSV,
    )
else:
    filter_comparison = pd.read_csv(FILTER_COMPARISON_CSV)
    required_columns = {
        "class", "set_name", "FID", "IS_mean", "precision", "recall", "density", "coverage"
    }
    missing_columns = required_columns - set(filter_comparison.columns)

    if missing_columns:
        print(
            "Il CSV non contiene ancora tutte le metriche richieste:",
            sorted(missing_columns),
        )
    else:
        filter_comparison["variant"] = filter_comparison["set_name"].map(
            lambda name: "Filtrate" if "_filtered_" in name else "RAW"
        )
        filter_classes = ["positive", "negative"]
        class_colors = {"positive": "#1f77b4", "negative": "#ff7f0e"}

        final_test_context = None
        if FILTER_FINAL_TEST_JSON.is_file():
            with FILTER_FINAL_TEST_JSON.open(encoding="utf-8") as handle:
                final_test_context = json.load(handle)
        else:
            print(
                "Metriche finali sul test non disponibili; "
                "i grafici useranno soltanto il validation:",
                FILTER_FINAL_TEST_JSON,
            )

        def metric_pair(class_name, metric):
            class_rows = filter_comparison.loc[filter_comparison["class"] == class_name]
            values = class_rows.set_index("variant")[metric]
            if not {"RAW", "Filtrate"} <= set(values.index):
                raise ValueError(
                    f"Confronto RAW/Filtrate incompleto per {class_name}, metrica {metric}."
                )
            return float(values["RAW"]), float(values["Filtrate"])

        # 1. Traiettorie prima/dopo: ogni metrica mantiene la propria scala.
        metric_specs = [
            ("FID", "FID - più basso è meglio"),
            ("IS_mean", "Inception Score - più alto indica maggiore varietà"),
            ("precision", "Precision PRDC - più alto è meglio"),
            ("recall", "Recall PRDC - più alto indica maggiore copertura"),
        ]
        figure, axes = plt.subplots(2, 2, figsize=(13, 10), constrained_layout=True)
        for axis, (metric, title) in zip(axes.flat, metric_specs):
            for class_name in filter_classes:
                raw_value, filtered_value = metric_pair(class_name, metric)
                actual_delta = (filtered_value - raw_value) / raw_value * 100
                axis.plot(
                    [0, 1],
                    [raw_value, filtered_value],
                    color=class_colors[class_name],
                    marker="o",
                    linewidth=2.2,
                    markersize=7,
                    label=class_name,
                )
                axis.annotate(
                    "",
                    xy=(1, filtered_value),
                    xytext=(0, raw_value),
                    arrowprops={
                        "arrowstyle": "-|>",
                        "color": class_colors[class_name],
                        "linewidth": 2.2,
                        "mutation_scale": 13,
                    },
                )
                axis.annotate(
                    f"{actual_delta:+.1f}%",
                    xy=(1, filtered_value),
                    xytext=(7, 0),
                    textcoords="offset points",
                    va="center",
                    fontsize=9,
                    color=class_colors[class_name],
                )

            axis.set_xticks([0, 1], ["RAW", "Filtrate"])
            axis.set_title(title)
            axis.grid(axis="y", alpha=0.25)

        axes[0, 0].legend(title="Classe")
        figure.suptitle(
            "Effetto del filtro adattivo: più fedeltà, con un trade-off sulla varietà",
            fontsize=15,
        )
        if final_test_context is not None:
            test_context_text = (
                "Contesto test sulle sole filtrate: "
                f"FID medio={final_test_context.get('avg_FID', float('nan')):.2f}, "
                f"precision media={final_test_context.get('avg_precision', float('nan')):.3f}, "
                f"recall media={final_test_context.get('avg_recall', float('nan')):.3f}"
            )
            figure.text(0.5, -0.015, test_context_text, ha="center", fontsize=9)
        show_and_save_filter_figure(figure, "filter_before_after.png")

        # 2. Radar PRDC: RAW e filtrate sono confrontabili sulla stessa scala.
        prdc_metrics = ["precision", "recall", "density", "coverage"]
        radar_angles = np.linspace(0, 2 * np.pi, len(prdc_metrics), endpoint=False).tolist()
        radar_angles += radar_angles[:1]
        figure, axes = plt.subplots(
            1,
            2,
            figsize=(12, 6),
            subplot_kw={"polar": True},
            constrained_layout=True,
        )
        for axis, class_name in zip(axes, filter_classes):
            class_rows = filter_comparison.loc[
                filter_comparison["class"] == class_name
            ].set_index("variant")
            raw_values = [float(class_rows.loc["RAW", metric]) for metric in prdc_metrics]
            filtered_values = [
                float(class_rows.loc["Filtrate", metric]) for metric in prdc_metrics
            ]
            radial_limit = max(1.0, max(raw_values + filtered_values) * 1.1)

            for label, values, color in [
                ("RAW", raw_values, "#777777"),
                ("Filtrate", filtered_values, class_colors[class_name]),
            ]:
                closed_values = values + values[:1]
                axis.plot(radar_angles, closed_values, color=color, linewidth=2, label=label)
                axis.fill(radar_angles, closed_values, color=color, alpha=0.18)

            axis.set_xticks(radar_angles[:-1], [name.capitalize() for name in prdc_metrics])
            axis.set_ylim(0, radial_limit)
            axis.set_title(class_name.capitalize(), pad=20)
            axis.legend(loc="lower right", bbox_to_anchor=(1.2, -0.1))

        figure.suptitle("Profilo PRDC prima e dopo il filtro adattivo", fontsize=15)
        show_and_save_filter_figure(figure, "filter_prdc_radar.png")

        # 3. Delta orientato: valori positivi indicano miglioramento della metrica.
        delta_metrics = ["FID", "IS_mean", "precision", "recall", "density", "coverage"]
        oriented_delta_rows = []
        for class_name in filter_classes:
            for metric in delta_metrics:
                raw_value, filtered_value = metric_pair(class_name, metric)
                raw_delta = (filtered_value - raw_value) / raw_value * 100
                oriented_delta = -raw_delta if metric == "FID" else raw_delta
                oriented_delta_rows.append({
                    "class": class_name,
                    "metric": metric,
                    "oriented_delta": oriented_delta,
                })

        delta_frame = pd.DataFrame(oriented_delta_rows)
        figure, axis = plt.subplots(figsize=(11, 7), constrained_layout=True)
        y_positions = np.arange(len(delta_metrics))
        bar_height = 0.34
        class_offsets = {"positive": -bar_height / 2, "negative": bar_height / 2}
        class_hatches = {"positive": "//", "negative": "\\"}

        for class_name in filter_classes:
            class_delta = (
                delta_frame.loc[delta_frame["class"] == class_name]
                .set_index("metric")
                .reindex(delta_metrics)["oriented_delta"]
            )
            positions = y_positions + class_offsets[class_name]
            colors = [
                "#2e8b57" if value >= 0 else "#c0392b"
                for value in class_delta
            ]
            bars = axis.barh(
                positions,
                class_delta,
                height=bar_height,
                color=colors,
                edgecolor="white",
                hatch=class_hatches[class_name],
            )
            axis.bar_label(bars, labels=[f"{value:+.1f}%" for value in class_delta], padding=3)

        axis.axvline(0, color="black", linewidth=1)
        axis.set_yticks(y_positions, ["FID (segno invertito)", "IS_mean", "Precision", "Recall", "Density", "Coverage"])
        axis.set_xlabel("Variazione percentuale orientata al miglioramento")
        axis.set_title("Delta del filtro: fedeltà guadagnata e varietà sacrificata")
        axis.grid(axis="x", alpha=0.25)
        axis.legend(handles=[
            Patch(facecolor="#2e8b57", label="Miglioramento"),
            Patch(facecolor="#c0392b", label="Riduzione / trade-off"),
            Patch(facecolor="white", edgecolor="black", hatch="//", label="Positive"),
            Patch(facecolor="white", edgecolor="black", hatch="\\", label="Negative"),
        ], loc="best")
        axis.text(
            0.5,
            -0.13,
            "Per il FID il segno è invertito: una barra positiva indica che il valore è diminuito. "
            "Le riduzioni di IS e recall descrivono il trade-off su varietà e copertura.",
            transform=axis.transAxes,
            ha="center",
            va="top",
            fontsize=9,
        )
        show_and_save_filter_figure(figure, "filter_oriented_percent_delta.png")

## 9. Test finale sulle immagini filtrate

L'ultima cella esegue la valutazione conclusiva sul test set, che non è stato utilizzato né per scegliere il checkpoint né per calibrare il filtro.

Per entrambe le classi vengono usate esclusivamente le 1361 immagini presenti nelle directory filtrate. Prima del calcolo vengono verificati il numero atteso di file e l'assenza di duplicati byte-per-byte. FID, Inception Score e PRDC sono quindi calcolati contro i reali del test della label corrispondente, riutilizzando le stesse feature Inception per FID e PRDC.

Il riepilogo include tutte le metriche per classe e le relative medie tra positive e negative, con riferimenti ai metadata reali e alle directory generate. Anche sul test, le metriche PRDC basate sui 73 riferimenti per classe sono più solide per confronti relativi che come stime assolute della distribuzione. I risultati vengono scritti in `results/03b_finetuning_filtered/metrics/final_test_metrics.json` e `results/03b_finetuning_filtered/metrics/final_test_metrics.csv`, sovrascrivendo in modo esplicito la precedente valutazione finale dello stesso schema.

Le metriche storiche calcolate sulle 2722 immagini RAW restano separate nei file `results/03b_finetuning_filtered/metrics/final_test_metrics_raw_2722.*`, così il confronto pre/post filtro non viene perso.


In [ ]:
final_test_metrics = {}
print("TEST FINALE: dataset filtrati contro i reali del test")
print("Checkpoint scelto sul validation:", BEST_CHECKPOINT.name)

for class_name in FINAL_GENERATE_CLASSES:
    label = FINAL_CLASS_LABELS[class_name]
    generated_dir = FILTERED_DIRS[class_name]
    n_generated = count_pngs(generated_dir)
    if n_generated != N_SELECTED_PER_CLASS:
        raise RuntimeError(
            f"Dataset filtrato incompleto per {class_name}: "
            f"{n_generated} != {N_SELECTED_PER_CLASS}."
        )
    if duplicate_png_groups(generated_dir):
        raise RuntimeError(f"Il dataset filtrato {class_name} contiene duplicati.")

    metrics = evaluate_generated_dir_against_split(
        generated_dir=generated_dir,
        metadata_path=TEST_METADATA_PATH,
        label=label,
        n_images=N_TEST_IMAGES_PER_CLASS,
        seed=EVAL_SEED + label,
    )
    final_test_metrics[class_name] = {
        **metrics,
        "real_label": label,
        "n_real_reference": N_TEST_IMAGES_PER_CLASS,
        "real_reference_metadata": str(TEST_METADATA_PATH),
        "generated_dir": str(generated_dir),
    }

n_classes = len(final_test_metrics)
average_metric_names = [
    "FID", "IS_mean", "IS_std", "precision", "recall", "density", "coverage"
]
averages = {
    f"avg_{metric}": round(
        sum(row[metric] for row in final_test_metrics.values()) / n_classes,
        4,
    )
    for metric in average_metric_names
}
final_test_summary = {
    "experiment": EXPERIMENT_NAME,
    "best_checkpoint": BEST_CHECKPOINT.name,
    "inference_steps": INFERENCE_STEPS,
    "evaluation_reference": "test",
    "classes_evaluated": list(final_test_metrics),
    **averages,
    "per_class": final_test_metrics,
    "timestamp": datetime.now().isoformat(timespec="seconds"),
}
with FINAL_TEST_METRICS_PATH.open("w", encoding="utf-8") as handle:
    json.dump(final_test_summary, handle, indent=2, ensure_ascii=False)

final_rows = [
    {"class": class_name, **metrics}
    for class_name, metrics in final_test_metrics.items()
]
final_rows.append({
    "class": "average",
    **{metric: averages[f"avg_{metric}"] for metric in average_metric_names},
    "n_generated": sum(row["n_generated"] for row in final_test_metrics.values()),
    "real_label": "",
    "n_real_reference": N_TEST_IMAGES_PER_CLASS,
    "real_reference_metadata": str(TEST_METADATA_PATH),
    "generated_dir": "",
})
df_final_test = pd.DataFrame(final_rows)
df_final_test.to_csv(FINAL_TEST_METRICS_CSV, index=False)

print("\nMetriche finali sul test:")
print(df_final_test.to_string(index=False))
print("\nSalvate in:", FINAL_TEST_METRICS_PATH, "e", FINAL_TEST_METRICS_CSV)

## 10. Confronto 50 vs 100 inference step

Questa sezione conclusiva legge esclusivamente le metriche già salvate dei due
esperimenti, senza generare immagini, ricalcolare metriche o usare la GPU:

- **50 step:** `experiments/20260607_sd21_rsna_mlo_512`;
- **100 step:** `results/03b_finetuning_filtered/metrics` per le metriche e
  `experiments/20260611_sd21_rsna_mlo_512_inference_100_steps` per immagini/modello.

L'esperimento storico a 50 step dispone soltanto del dataset finale **filtrato** della
classe positiva: 1361 immagini valutate nel relativo `final_test_metrics.json`. Non sono
disponibili né la classe negativa né un dataset RAW a 50 step. Il solo confronto omogeneo
possibile usa quindi la classe positiva finale filtrata in entrambi gli esperimenti, allo
stesso stadio e con la stessa numerosità:

- positive filtrate, 50 step: FID circa 118.98 e IS circa 2.708;
- positive filtrate, 100 step: FID circa 113.03 e IS circa 2.419.

Questo confronto isola l'effetto del numero di inference step **a parità di stadio di
filtraggio**. La riduzione del FID positive da circa 118.98 a 113.03, pari a circa il 5%,
è quindi associabile all'uso di 100 inference step. Il calo dell'Inception Score va letto
come variazione della varietà e non contraddice automaticamente il miglioramento del FID.

Il guadagno dovuto al filtro adattivo è una variabile distinta ed è analizzato nella
sezione precedente. Un confronto sulle negative o un confronto RAW-vs-RAW tra 50 e 100
step non sono possibili con gli artefatti disponibili e restano una limitazione, oltre che
un possibile lavoro futuro. La tabella include comunque le negative filtrate e i RAW dei
100 step come **contesto separato**, chiaramente etichettato e non usato nei grafici
comparativi.


In [ ]:
# Confronto in sola lettura: positive filtrate 50 vs 100 step.
EXPERIMENT_50_DIR = PROJECT_ROOT / "experiments" / "20260607_sd21_rsna_mlo_512"
INFERENCE_COMPARISON_CSV = METRICS_DIR / "comparison_50_vs_100_summary.csv"


def load_saved_metrics(path, description, fallback_path=None):
    path = Path(path)
    if not path.is_file() and fallback_path is not None:
        fallback_path = Path(fallback_path)
        if fallback_path.is_file():
            path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(fallback_path, path)
            print(f"{description}: copiato artefatto legacy in results: {path}")
    if not path.is_file():
        print(f"{description} non disponibile: {path}")
        return None
    try:
        with path.open(encoding="utf-8") as handle:
            return json.load(handle)
    except (OSError, json.JSONDecodeError) as exc:
        print(f"{description} non leggibile ({path}): {exc}")
        return None


def rows_from_payload(inference_steps, artifact, stage, payload, comparable_classes):
    rows = []
    if not payload:
        return rows
    for class_name, metrics in payload.get("per_class", {}).items():
        rows.append({
            "inference_steps": inference_steps,
            "artifact": artifact,
            "stadio": stage,
            "class": class_name,
            "comparabile_50_vs_100": class_name in comparable_classes and stage == "filtrato",
            "FID": metrics.get("FID"),
            "IS_mean": metrics.get("IS_mean"),
            "IS_std": metrics.get("IS_std"),
            "precision": metrics.get("precision"),
            "recall": metrics.get("recall"),
            "density": metrics.get("density"),
            "coverage": metrics.get("coverage"),
            "n_generated": metrics.get("n_generated"),
        })
    return rows


def positive_filtered_metric(payload, metric):
    if not payload:
        return None
    value = payload.get("per_class", {}).get("positive", {}).get(metric)
    return float(value) if value is not None else None


def plot_positive_filtered_comparison(metric, ylabel, title, payloads_by_step, output_name):
    values = [positive_filtered_metric(payloads_by_step.get(step), metric) for step in (50, 100)]
    if any(value is None for value in values):
        print(
            f"Confronto positive filtrate incompleto per {metric}: grafico non generato."
        )
        return

    figure, axis = plt.subplots(figsize=(8, 6), constrained_layout=True)
    steps = [50, 100]
    metric_values = values
    colors = ["#1f77b4" if step == 50 else "#ff7f0e" for step in steps]
    bars = axis.bar([str(step) for step in steps], metric_values, color=colors, width=0.55)
    axis.bar_label(bars, labels=[f"{value:.4f}" for value in metric_values], padding=4)
    axis.set_xlabel("Inference step")
    axis.set_ylabel(ylabel)
    axis.set_title(title)
    axis.grid(axis="y", alpha=0.25)
    figure.savefig(PLOTS_DIR / output_name, dpi=180, bbox_inches="tight")
    plt.show()
    plt.close(figure)
    print("Salvato:", PLOTS_DIR / output_name)


final_50 = load_saved_metrics(
    EXPERIMENT_50_DIR / "final_test_metrics.json",
    "Metriche finali filtrate 50 step",
)
final_100 = load_saved_metrics(
    FINAL_TEST_METRICS_PATH,
    "Metriche finali filtrate 100 step",
)
raw_100 = load_saved_metrics(
    FINAL_TEST_RAW_METRICS_PATH,
    "Metriche RAW 100 step usate solo come contesto",
    fallback_path=LEGACY_FINAL_TEST_RAW_METRICS_PATH,
)

positive_50 = final_50.get("per_class", {}).get("positive") if final_50 else None
positive_100 = final_100.get("per_class", {}).get("positive") if final_100 else None
comparison_is_homogeneous = (
    positive_50 is not None
    and positive_100 is not None
    and positive_50.get("n_generated") == positive_100.get("n_generated")
)
comparable_classes = {"positive"} if comparison_is_homogeneous else set()
comparison_rows_50_100 = [
    *rows_from_payload(
        50,
        "final_test_metrics",
        "filtrato",
        final_50,
        comparable_classes,
    ),
    *rows_from_payload(
        100,
        "final_test_metrics",
        "filtrato",
        final_100,
        comparable_classes,
    ),
    *rows_from_payload(
        100,
        "final_test_metrics_raw_2722",
        "raw",
        raw_100,
        comparable_classes,
    ),
]

if comparison_rows_50_100:
    df_comparison_50_100 = pd.DataFrame(comparison_rows_50_100)
    comparison_columns = [
        "inference_steps", "artifact", "stadio", "class", "comparabile_50_vs_100",
        "FID", "IS_mean", "IS_std", "precision", "recall", "density", "coverage",
        "n_generated",
    ]
    df_comparison_50_100 = df_comparison_50_100.reindex(columns=comparison_columns)
    df_comparison_50_100.to_csv(INFERENCE_COMPARISON_CSV, index=False)
    print("\nRiepilogo degli artefatti disponibili:")
    print(df_comparison_50_100.to_string(index=False))
    print("\nLe righe RAW dei 100 step sono riportate solo come contesto.")
    print("Riepilogo salvato in:", INFERENCE_COMPARISON_CSV)
else:
    print("Nessuna metrica finale disponibile per il riepilogo 50 vs 100 step.")

if comparison_is_homogeneous:
    fid_50 = float(positive_50["FID"])
    fid_100 = float(positive_100["FID"])
    fid_delta_percent = (fid_100 - fid_50) / fid_50 * 100
    print(
        "\nConfronto valido: positive finali filtrate, "
        f"{positive_50['n_generated']} immagini per esperimento."
    )
    print(
        f"FID positive: {fid_50:.4f} (50 step) -> {fid_100:.4f} (100 step), "
        f"variazione {fid_delta_percent:+.1f}%."
    )
    print(
        "Per i 50 step non sono disponibili negative né RAW: "
        "questi confronti non sono possibili e restano lavoro futuro."
    )
else:
    print(
        "Confronto positive filtrate 50 vs 100 non completo o con numerosità diverse; "
        "i grafici mostreranno soltanto i valori omogenei disponibili."
    )

filtered_payloads_by_step = {50: final_50, 100: final_100}
plot_positive_filtered_comparison(
    metric="FID",
    ylabel="FID - più basso è meglio",
    title="FID positive finali filtrate: 50 vs 100 inference step",
    payloads_by_step=filtered_payloads_by_step,
    output_name="comparison_50_vs_100_fid.png",
)
plot_positive_filtered_comparison(
    metric="IS_mean",
    ylabel="Inception Score",
    title="Inception Score positive finali filtrate: 50 vs 100 inference step",
    payloads_by_step=filtered_payloads_by_step,
    output_name="comparison_50_vs_100_is.png",
)